# Multi AI Agent workflow

### Architecture Overview
- Team of AI agents that can work together to achieve a common goal.
- **Reasercher Agent**: Responsible for gathering and analyzing data relevant to the task. Agent will use tavil to search for information, summarize findings, and provide insights to the team.
- **Analyst Agent**: Responsible for interpreting the data provided by the Researcher Agent. This agent will identify patterns, trends, and potential solutions based on the research findings.
- **Writer Agent**: Responsible for documenting the findings and solutions proposed by the Analyst Agent. This agent will create reports, presentations, and other written materials to communicate the team's work effectively.
- **Manager function**: Oversees the workflow of the AI agents, ensuring that tasks are completed efficiently and effectively. The manager will coordinate the efforts of the Researcher, Analyst, and Writer agents, providing guidance and support as needed.

#### Architecture Diagram - Multi AI Agent Workflow

```mermaid
flowchart TD
    U([User Query]) --> M

    subgraph M["Manager: manager_run() + SQLiteSession (shared memory)"]
        direction TB
        R["Researcher Agent<br/>out: AnalysisSummary"]
        A["Analyst Agent<br/>out: AnalysisSummary"]
        W["Writer Agent<br/>out: FinalReport"]
        R -->|research summary| A
        A -->|analysis summary| W
    end

    R <-->|tool call| T["tavily_search<br/>@function_tool<br/>POST api.tavily.com/search"]
    W --> O["print_markdown()<br/>short_summary, markdown_report,<br/>follow_up_questions"]

    S["Setup: .env / getpass<br/>OPENAI_API_KEY, TAVILY_API_KEY<br/>OpenAI Agents SDK + Pydantic"] -.-> M
```

#### Architecture Diagram (text fallback)

```text
                      +----------------------+
                      |   User Query         |
                      +----------+-----------+
                                 |
                                 v
        +--------------------------------------------------+
        |        MANAGER  -  manager_run()  (async)        |
        |  orchestrates agents via Runner.run(...)         |
        |  shared memory: SQLiteSession                    |
        +----+---------------+----------------+------------+
             | 1. query      | 2. research    | 3. query + research
             v               v    summary     v    + analysis
   +------------------+  +------------------+  +-----------------------+
   | RESEARCHER agent |  |  ANALYST agent   |  |     WRITER agent      |
   | out: Analysis-   |  | out: Analysis-   |  | out: FinalReport      |
   |      Summary     |  |      Summary     |  |  - short_summary      |
   +--------+---------+  | (trends, risks,  |  |  - markdown_report    |
            |            |  insights)       |  |  - follow_up_questions|
            |            +------------------+  +----------+------------+
            | tool call
            v                                             |
   +------------------+                                   v
   |  tavily_search   |                        +----------------------+
   | @function_tool   |                        |  print_markdown()    |
   | POST api.tavily  |                        |  Report shown to user|
   |  .com/search     |                        +----------------------+
   +------------------+

 Flow:  User -> Manager -> Researcher (+Tavily) -> Analyst -> Writer -> Report

 Setup:  .env / getpass -> OPENAI_API_KEY, TAVILY_API_KEY
         OpenAI Agents SDK (Agent, Runner, function_tool, SQLiteSession)
         Pydantic models enforce structured outputs; LLM = gpt_model
```

### Setup tools and libraries

In [1]:
!pip install -q openai-agents==0.2.2 python-dotenv requests pydantic

In [2]:
import os
import requests
from IPython.display import display, Markdown
from openai import OpenAI
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool, SQLiteSession
from typing_extensions import TypedDict
from pydantic import BaseModel

In [3]:
# Setup env. and Load API keys from .env file
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

if not openai_api_key:
    from getpass import getpass
    openai_api_key = getpass("OpenAI API key (will not be echoed): ")
    
# Ensure the agents/OpenAI client can read the key via the environment variable
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

if not tavily_api_key:
    from getpass import getpass
    tavily_api_key = getpass("Tavily API key (will not be echoed): ")

# Ensure the agents/OpenAI client can read the key via the environment variable
if tavily_api_key:
    os.environ["TAVILY_API_KEY"] = tavily_api_key

openai_client = OpenAI(api_key=openai_api_key)

print("✅ API keys loaded")
print(f"  OpenAI … {openai_api_key[:5]}***")
print(f"  Tavily … {tavily_api_key[:5]}***")

✅ API keys loaded
  OpenAI … sk-pr***
  Tavily … tvly-***


In [19]:
gpt_model = "gpt-6.1-terra"

### Setup Tavily Search Tool and Markdown class

In [5]:
# Define a function to display markdown in Jupyter Notebook
def print_markdown(text):
    display(Markdown(text))

In [6]:
# A TypedDict describes the expected keys and value types for a dict.
# Here: a search "query" (string) and "max_results" (int).
class TavilySearchParams(TypedDict):
    query: str
    max_results: int

In [7]:
# This decorator (from OpenAI Agents SDK) registers the function as a tool
# that an agent can call. 

@function_tool
def tavily_search(params: TavilySearchParams) -> str:

    # Tavily search endpoint
    url = "https://api.tavily.com/search"

    # Tell the API we're sending JSON
    headers = {"Content-Type": "application/json"}

    # Build the request body:
    # api_key: your Tavily API key (assumed defined elsewhere as tavily_api_key)
    # query: taken from the params dict
    # max_results: use provided value or default to 3 if missing
    payload = {
        "api_key": tavily_api_key,
        "query": params["query"],
        "max_results": params.get("max_results", 3),
    }

    # Send the POST request with JSON body and headers
    response = requests.post(url, json = payload, headers = headers)
    if response.status_code == 200:
        results = response.json().get("results", [])
        summary = "\n".join([f"- {r['title']}: {r['content']}" for r in results])
        print_markdown(summary if summary else "No relevant results found.")
        return summary if summary else "No relevant results found."
    else:
        return f"Tavily API error: {response.status_code}"

print("✅ Tavily search tool ready.")

✅ Tavily search tool ready.


### Researcher AI agent - Define and Execute

In [8]:
# Let's define our Researcher AI Agent

# Define a Pydantic model for the agent's output.
# This ensures the agent's final output will have a single field: "summary" (string).
# Note that this will be used in both the researcher and analyst agents
class AnalysisSummary(BaseModel):
    summary: str

In [20]:
# Create an AI agent called "Researcher"
researcher_agent = Agent(
    name = "Researcher",
    instructions = """
        ## Context
        You are a research agent with access to the Tavily search tool.

        ## Instruction
        Given a user query, use the Tavily search tool to find relevant information and summarize the key findings.

        ## Input
        - A research query from the user.

        ## Output
        - A summary of key findings in a maximum of 5 bullet points.
        """,
    model = gpt_model,
    tools = [tavily_search],
    output_type = AnalysisSummary
)

print("✅ Researcher AI agent is now ready")

✅ Researcher AI agent is now ready


In [10]:
# Let's run the researcher AI agent
researcher_result = await Runner.run(researcher_agent, "Why is Little Mermaid so popular in Denmark and what is the history behind it?")
researcher_result

- The Little Mermaid (statue) - Wikipedia: Based on the 1837 fairy tale of the same name by Danish author Hans Christian Andersen, the small and unimposing statue is a Copenhagen icon and has been a major tourist attraction since its unveiling in 1913. In recent decades it has become a popular target for defacement by vandals and political activists. [...] The Little Mermaid (Danish: Den lille Havfrue) is a bronze statue by Edvard Eriksen, depicting a mermaid becoming human. The sculpture is displayed on a rock by the waterside at the Langelinie promenade in Copenhagen, Denmark. It is 1.25 metres (4.1 ft) tall and weighs 175 kilograms (385 lb). [...] | Coordinates | 55°41′34.3″N 12°35′57.4″E﻿ / ﻿55.692861°N 12.599278°E﻿ / 55.692861; 12.599278&params=55_41_34.3_N_12_35_57.4_E_type:landmark_region:DK-84): {{#parsoid\u0000fragment:6}}[ 55°41′34.3″N 12°35′57.4″E﻿ / ﻿55.692861°N 12.599278°E﻿ / 55.692861; 12.599278]"}}" id="mwCw" about="#mwt9" typeof="mw:Extension/indicator"> |
| Designer | Edvard Eriksen |
| Type | Statue |
| Material | Bronze |
| Opening date | August 23, 1913 |
- Little Mermaid Statue Copenhagen - History, Location & Vi...: The statue was unveiled on August 23, 1913, and was commissioned by Carl Jacobsen, the founder of Carlsberg Breweries, after he became enchanted by a ballet performance of Hans Christian Andersen's 1837 fairy tale. The sculptor, Edvard Eriksen, originally wanted the ballerina Ellen Price to model for the figure, but she declined to pose nude. Eriksen's wife, Eline, became the model for the body, while Price's face was used for the head. [...] The Little Mermaid is Copenhagen's most famous landmark, a bronze statue sitting on a rock at the Langelinie promenade, gazing out over the harbour. [...] The statue has had a turbulent history. It has been vandalized repeatedly: its head has been sawn off twice (1964 and 1998), its arm was removed in 1984, and it has been covered in paint, dressed in a burqa, and adorned with various protest messages over the decades. Each time, it has been restored.

In 2010, the statue was temporarily relocated to the Danish Pavilion at the Shanghai World Expo, the first time it had left Copenhagen.
- Little Mermaid Statue Copenhagen: How to Visit & History: | Fact | Detail |
 --- |
| Danish name | Den lille Havfrue |
| Location | Langelinie, 2100 København Ø |
| Unveiled | 23 August 1913 |
| Sculptor | Edvard Eriksen |
| Commissioned by | Carl Jacobsen (Carlsberg brewery, 1909) |
| Inspiration | Hans Christian Andersen’s 1837 fairy tale ‘Den lille Havfrue’ |
| Material | Bronze on a granite boulder |
| Height | 125 cm (4 ft 1 in) — much smaller than visitors expect |
| Weight | 175 kg | [...] The Little Mermaid statue (Den Lille Havfrue) sits on Langelinie promenade — bronze, 125 cm tall, modelled in 1913 by Edvard Eriksen for brewer Carl Jacobsen.

 

Serene view of a mermaid statue perched on a rock in a vast ocean landscape — Copenhagen's Little Mermaid statue is the most photographed sculpture in Denmark

The Little Mermaid statue (Den Lille Havfrue) sits on Langelinie promenade — bronze, 125 cm tall, modelled in 1913 by Edvard Eriksen for brewer Carl Jacobsen. [...] Skip to content

Copenhagen   
Tourism Guide .org

Things to Do

# Little Mermaid Statue Copenhagen: Complete Visitor Guide for 2026

An honest guide to the Little Mermaid statue in Copenhagen: history, how to get there, the best harbour walk and photo times, and if she's worth it.

14 min read Copenhagen Tourism

Serene view of a mermaid statue perched on a rock in a vast ocean landscape — Copenhagen's Little Mermaid statue is the most photographed sculpture in Denmark
- The Little Mermaid | Iconic tourist attraction in Copenhagen: Walking down Langelinie Pier, you’ll find The Little Mermaid gracefully sitting on a rock in the water, gazing longingly out over the shore. The sculpture was made in 1913 by Edvard Eriksen and was gifted by the Danish brewer Carl Jacobsen. Since then, The Little Mermaid has become one of Copenhagen’s most iconic landmarks and a city symbol. If you know the timeless fairy tale by Hans Christian Andersen, you’ll immediately recognize the connection – the Little Mermaid sculpture is inspired by [...] Carl Jacobsen’s fascination began after watching a ballet version of The Little Mermaid. Captivated by both the story and the performance, he commissioned Edvard Eriksen to bring the mermaid to life in sculpture. Ellen Price, who danced the lead role in 1909, inspired the face but declined to model nude. Instead, Eriksen’s wife Eline posed for the body. The mermaid has since become a global icon of Copenhagen, despite repeated vandalism. Her resilience and continual restoration reflect the [...] The mermaid was sculpted by Edvard Eriksen. The face was inspired by ballerina Ellen Price, while the body was modelled by Eriksen’s wife, Eline.

#### Why is she linked to Hans Christian Andersen?

The sculpture was inspired by Andersen’s 1837 fairy tale about a mermaid who gives up her underwater life for love. Jacobsen was moved by a ballet performance of the story at the Royal Danish Theatre.

#### Has The Little Mermaid ever been vandalised?
- Denmark’s Leading Lady: Odd Facts About the Little Mermaid - YMT Vacations: Because of the popularity of her story, Copenhagen’s Little Mermaid has gotten a lot of attention over the years. Here are just a few of our favorite The Little Mermaid facts:

1830 – Hans Christian Andersen writes his most popular fairytale—a melancholy story about art, longing, and self-denial set in the tumultuous undersea world of mermaids and mermen.

1913 – Wealthy Copenhagen brewer Carl Jacobsen commissions local sculptor Edvard Eriksen to create a statue of the story’s heroine. [...] A beautiful, wistful young lady astride a submerged rock just off Copenhagen Harbor, The Little Mermaid statue is one of Denmark’s most popular tourist attractions. Mostly beloved, she has lived an exciting 100+ years since first being sculpted early in the 20th century.

The famous statue is based on a character from an 1830 Hans Christian Andersen fairytale. The youngest daughter of the widowed Sea King yearns for a chance at love with the handsome human prince she saves from drowning. [...] Carl Jacobsen, a local brewer, was so touched by a ballet performance of the story he saw at the Royal Danish Theatre that he commissioned The Little Mermaid statue, Edvard Eriksen’s masterwork. The famous Disney film version of The Little Mermaid is based on the original story, but has a significantly happier ending—unsurprising given the characteristic tragedy and horror found throughout Anderson’s authentic fairytale.

## The Little Mermaid’s Timeline

- The Little Mermaid (statue) - Wikipedia: Based on the 1837 fairy tale of the same name by Danish author Hans Christian Andersen, the small and unimposing statue is a Copenhagen icon and has been a major tourist attraction since its unveiling in 1913. In recent decades it has become a popular target for defacement by vandals and political activists. [...] The statue was commissioned in 1909 by Carl Jacobsen, son of the founder of Carlsberg, who had been fascinated by a ballet about the fairytale in Copenhagen's Royal Theatre and asked the ballerina, Ellen Price, to model for the statue. [...] The Little Mermaid (Danish: Den lille Havfrue) is a bronze statue by Edvard Eriksen, depicting a mermaid becoming human. The sculpture is displayed on a rock by the waterside at the Langelinie promenade in Copenhagen, Denmark. It is 1.25 metres (4.1 ft) tall and weighs 175 kilograms (385 lb).
- The Little Mermaid Statue in Copenhagen Denmark the iconic tourist attraction: The Little Mermaid (Den lille Havfrue) is a bronze statue by Edvard Eriksen, depicting a mermaid becoming human. The sculpture is displayed on a rock by the waterside at the Langelinie promenade in Copenhagen, Denmark. It is 1.25 metres (4.1 ft) tall and weighs 175 kilograms (385 lb). Based on the 1837 fairy tale of the same name by Danish author Hans Christian Andersen, the small and unimposing statue is a Copenhagen icon and has been a major tourist attraction since its unveiling in 1913. [...] ### Description
2,504 views
Posted: 2024-12-01
Join me in Copenhagen, Denmark, as we explore the iconic tourist attraction, the Little Mermaid statue. Learn about some of the history of this beloved statue, known as Den Lille Havfrue in Danish, and get a glimpse of what to expect when you visit. Watch now to immerse yourself in the beauty of Copenhagen's famous landmark. [...] The Little Mermaid
- Little Mermaid Statue Copenhagen - History, Location & Vi...: The statue was unveiled on August 23, 1913, and was commissioned by Carl Jacobsen, the founder of Carlsberg Breweries, after he became enchanted by a ballet performance of Hans Christian Andersen's 1837 fairy tale. The sculptor, Edvard Eriksen, originally wanted the ballerina Ellen Price to model for the figure, but she declined to pose nude. Eriksen's wife, Eline, became the model for the body, while Price's face was used for the head. [...] The Little Mermaid is Copenhagen's most famous landmark, a bronze statue sitting on a rock at the Langelinie promenade, gazing out over the harbour. [...] The statue has had a turbulent history. It has been vandalized repeatedly: its head has been sawn off twice (1964 and 1998), its arm was removed in 1984, and it has been covered in paint, dressed in a burqa, and adorned with various protest messages over the decades. Each time, it has been restored.

In 2010, the statue was temporarily relocated to the Danish Pavilion at the Shanghai World Expo, the first time it had left Copenhagen.
- The Little Mermaid of Copenhagen: A Must-See Landmark in Denmark | Grand European Travel: The statue was unveiled in 1913 as a tribute to Andersen’s beloved story. Commissioned by Danish brewer Carl Jacobsen, it was sculpted by Edvard Eriksen. The model for the mermaid's face was ballet dancer Ellen Price, but the body was modeled after Eriksen's wife, Eline. Over the years, the statue has become a powerful symbol of Copenhagen and Danish culture.

### The Symbolism of The Little Mermaid [...] ## Discovering The Little Mermaid Statue

The Little Mermaid statue in Copenhagen is one of Denmark's most famous landmarks. Inspired by Hans Christian Andersen’s fairy tale, this iconic bronze sculpture has been captivating visitors for over a century. Situated along the Langelinie promenade, the statue offers a glimpse into Denmark’s rich literary and artistic heritage.

### The History Behind The Little Mermaid
- Copenhagen’s Little Mermaid: Fairy-Tale or Tourist Trap? - The Hidden North: The Little Mermaid statue was commissioned by Carl Jacobsen of the Carlsberg brewing family in 1909 after he saw a ballet of Andersen’s tale (or tail, ha! Sorry, that was bad). He asked ballerina Ellen Price (who had danced the role in 1909) to model for the sculpture’s head. Sculptor Edvard Eriksen created the bronze figure, using his wife for the body after Price declined to pose nude. The statue, which is only 1.25 metres tall, was unveiled on the 23rd of August 1913 on a rock at Langelinie [...] was a Danish writer best known for his fairy tales. In 1837, he published The Little Mermaid (“Den lille havfrue” in Danish), telling of a young mermaid princess who dreams of a human soul and falls in love with a prince above the sea. In the original fairytale, which is very different from Disney’s happy ending, the mermaid saves the princes from a shipwreck and sacrifices her voice and life to become human. The prince never loves her back; on his wedding day, her sisters give the heartbroken

- The Little Mermaid - Wikipedia: "The Little Mermaid" (Danish: Den lille havfrue), sometimes translated in English as "The Little Sea Maid", is a literary fairy tale written by Danish author Hans Christian Andersen. Originally published in 1837 as part of a collection of fairy tales for children, the story follows the journey of a young mermaid princess who is willing to give up her life in the sea as a mermaid to gain a human soul. [...] The statue was commissioned in 1909 by Carl Jacobsen, son of the founder of Carlsberg, after he had been fascinated by a ballet based on the fairy tale. The sculptor Edward Eriksen created the statue, which was unveiled on 23 August 1913. His wife, Eline Eriksen, was the model for the body. Ellen Price, the ballerina who danced the Little Mermaid in the 1909 Royal Danish Ballet production, was the model for the head and face. The statue has been severely vandalized several times. [...] | Short story by Hans Christian Andersen |
| The Little Mermaid – illustration by Edmund Dulac |
| Text available at Wikisource |
| Text available at Danish Wikisource |
| Original title | Den lille havfrue |
| Translator | Mary Howitt |
| Country | Denmark |
| Language | Danish |
| Genres | Fantasy, fairy tale |
| Publication |
| Published in | Fairy Tales Told for Children. First Collection. Third Booklet. 1837. (Eventyr, fortalte for Børn. Første Samling. Tredie Hefte. 1837.) |
- The Little Mermaid | Folk and Fairy Tales | Pook Press: Andersen was a writer of many forms: novels, plays and poetry, but his fairy tales are well-known and loved by people across the world. The charming, original story of The Little Mermaid was published in Copenhagen by C.A. Reitzel on 7th April 1837, in the first collection of Fairy Tales Told for Children. [...] Another widely known adaptation of the fairy tale is the Royal Danish Ballet’s production. Carl Jacobsen, son of the founder of Carlsberg, went to see the performance at the Royal Danish Theatre in Copenhagen. Jacobsen was enthralled by the ballet and he commissioned Edvard Eriksen to sculpt a statue of the mermaid. The sculpture was based on Ellen Price, who danced the lead role of the ballet in 1909. The statue was unveiled on 23rd August 1913 and was a gift from Jacobsen to Copenhagen. It is [...] Hans Christian Andersen wrote the fairy tale, The Little Mermaid, in 1836. Andersen was a Danish author, born in Odense, Denmark on the 2nd of April 1805. He had a difficult start in life, being born to poor parents and growing up in a one-bedroom home. His father was ill and passed away when Andersen was only eleven, forcing his mother to send him away to earn money. He was an apprentice weaver, then worked in a tailor’s and a tobacco factory.
- The Little Mermaid statue in Copenhagen, Denmark — Mermaids of Earth: Eriksen was commissioned in January 1909 by Carl Jacobsen of Carlsberg Breweries to create the statue. Carl was fascinated by a ballet at the Copenhagen Royal Theatre based on the fairy tale about the mermaid, and asked the star of the ballet, Ellen Price de Plane, to model for the statue.  Price declined modeling in the nude for the sculpture, and Eriksen instead enlisted his wife Eline Eriksen (who modeled for several other of his works) to model for the mermaid statue.   A popular story has [...] The statue depicts the heroine of Hans Christian Andersen's 1837 fairy tale, a story considerably darker than most modern retellings. To gain human legs and a chance at the prince's love, the mermaid gives up her voice and tongue to the sea witch, and is warned that every step will feel like walking on knives. She is also told that if the prince marries another, she will dissolve into sea foam. When he does marry another, her sisters bring her a knife, bought from the witch with their hair, and [...] The Little Mermaid Statue in Copenhagen

The Little Mermaid Statue in Copenhagen. Photo by Rodolfo Puig.

The Little Mermaid - Commemorative Edition

The Little Mermaid - Commemorative Edition

Mermaids Of Earth Coffee-table book

Mermaids Of Earth coffee-table book

The mermaid statue was created in bronze by Edvard Eriksen, and was unveiled in August of 1913.
- A tale of two mermaids: Den Lille Havfrue, as she is known in Danish, was sculpted by Danish-Icelandic artist Edvard Eriksen and first unveiled to the public on 23rd August 1913. Commissioned by brewer Carl Jacobsen after he was captivated by a ballet performance of the famous fairytale, the Little Mermaid we see today is modelled on the face of Ellen Price and the body of the sculptor’s wife, Eline, as the former wished not to pose in the nude. She may not seem imposing at first glance, but she is the embodiment of a [...] On the edge of Copenhagen’s harbour sits a delicate bronze figure gazing wistfully out towards the ocean, immortalising Hans Christian Andersen’s famous tale of love and loss – the Little Mermaid. But few visitors realise that, just a short walk away, is her sister – distorted, defiant and unmistakably modern. On Langelinie Allé sits the Genetically Modified Little Mermaid, serving as a provocative counterpoint to her famous fairytale sibling, reflecting the complicated beauty and unease of our [...] The Little Mermaid
- The Little Mermaid | Iconic tourist attraction in Copenhagen: The mermaid was sculpted by Edvard Eriksen. The face was inspired by ballerina Ellen Price, while the body was modelled by Eriksen’s wife, Eline.

#### Why is she linked to Hans Christian Andersen?

The sculpture was inspired by Andersen’s 1837 fairy tale about a mermaid who gives up her underwater life for love. Jacobsen was moved by a ballet performance of the story at the Royal Danish Theatre.

#### Has The Little Mermaid ever been vandalised? [...] Walking down Langelinie Pier, you’ll find The Little Mermaid gracefully sitting on a rock in the water, gazing longingly out over the shore. The sculpture was made in 1913 by Edvard Eriksen and was gifted by the Danish brewer Carl Jacobsen. Since then, The Little Mermaid has become one of Copenhagen’s most iconic landmarks and a city symbol. If you know the timeless fairy tale by Hans Christian Andersen, you’ll immediately recognize the connection – the Little Mermaid sculpture is inspired by [...] Carl Jacobsen’s fascination began after watching a ballet version of The Little Mermaid. Captivated by both the story and the performance, he commissioned Edvard Eriksen to bring the mermaid to life in sculpture. Ellen Price, who danced the lead role in 1909, inspired the face but declined to model nude. Instead, Eriksen’s wife Eline posed for the body. The mermaid has since become a global icon of Copenhagen, despite repeated vandalism. Her resilience and continual restoration reflect the

RunResult(input='Why is Little Mermaid so popular in Denmark and what is the history behind it?', new_items=[ToolCallItem(agent=Agent(name='Researcher', handoff_description=None, tools=[FunctionTool(name='tavily_search', description='', params_json_schema={'$defs': {'TavilySearchParams': {'properties': {'query': {'title': 'Query', 'type': 'string'}, 'max_results': {'title': 'Max Results', 'type': 'integer'}}, 'required': ['query', 'max_results'], 'title': 'TavilySearchParams', 'type': 'object', 'additionalProperties': False}}, 'properties': {'params': {'$ref': '#/$defs/TavilySearchParams'}}, 'required': ['params'], 'title': 'tavily_search_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x780181093740>, strict_json_schema=True, is_enabled=True)], mcp_servers=[], mcp_config={}, instructions='\n        ## Context\n        You are a research agent with access to the Tavily search tool

In [11]:
print_markdown(f"### 🤖 Agent's Answer\n{researcher_result.final_output.summary}")

### 🤖 Agent's Answer
The Little Mermaid is popular in Denmark because it brings together a famous Danish fairy tale, a recognizable Copenhagen landmark, and a story that has reached audiences worldwide. A little history:

- **It began as a story, not a statue.** Danish writer Hans Christian Andersen published *The Little Mermaid* (*Den lille havfrue*) in 1837. The original is more melancholy than Disney’s version: the mermaid gives up her voice to become human, but the prince does not return her love.
- **A ballet inspired the statue.** In 1909, brewer and arts patron Carl Jacobsen saw a ballet based on Andersen’s tale and commissioned sculptor Edvard Eriksen to create a mermaid for Copenhagen. The bronze statue was unveiled at the Langelinie waterfront on 23 August 1913.
- **It has a human model’s story, too.** Ballerina Ellen Price, who danced the mermaid in the ballet, inspired the statue’s face. She declined to pose for the full figure, so Eriksen’s wife, Eline, modelled for its body.
- **Why it became so iconic:** The statue gives Copenhagen a compact, photogenic symbol tied to one of Denmark’s best-known writers. Its waterside setting and long history as a visitor attraction have made it a familiar emblem of the city—and, by extension, Danish culture. It is especially famous with visitors, though its modest size often surprises them.
- **It has stayed in the public eye.** The statue has been repeatedly vandalized and restored, and was temporarily moved to Shanghai for the 2010 World Expo. Copenhagen’s official tourism site links it directly to Andersen’s tale and describes it as one of the city’s iconic attractions.

### Analyst AI agent - Define and Execute 

In [21]:
# Let's define our Analyst AI Agent
analyst_agent = Agent(
    name = "Analyst",
    instructions = """
        ## Context
        You are an analyst who receives research notes generated by the research agent.

        ## Instruction
        Given the research notes, analyze the content and extract key trends, risks, or insights.

        ## Input
        - Research notes (summaries of findings from the research agent).

        ## Output
        - A concise analysis (no more than 2 paragraphs) highlighting key trends, risks, or insights.
        """,
    model = gpt_model,
    output_type = AnalysisSummary,
)

print("✅ Analyst AI agent is now ready")

✅ Analyst AI agent is now ready


### Writer AI agent - Define and Execute 
**Finale Executive Summary Report Generation**

In [13]:
# Define a Pydantic model for the agent's output
# This specifies exactly what fields the Writer agent should return
class FinalReport(BaseModel):
    short_summary: str                  # A brief 2–3 sentence executive summary
    markdown_report: str                # A detailed report in Markdown format (at least 500 words)
    follow_up_questions: list[str]      # A list of 3–5 suggested follow-up research questions

In [14]:
# Create an AI agent called "Writer"
writer_agent = Agent(
    name = "Writer",
    instructions = """
        You are a senior market analyst tasked with generating an executive-level research report.

        ## Context
        You will receive:
        - The original user query (the research question or topic of interest).
        - Summaries and analyses generated by helper tools: 'researcher_agent' (for bullet-point research findings) and 'analyst_agent' (for key trends, risks, or insights).

        You may call these helper tools as needed to gather additional information or clarify findings.

        ## Instructions
        Your job is to synthesize all available information and produce the following outputs:
        1. **Executive Summary:** Write a concise summary (2-3 sentences) highlighting the most important findings relevant to the original query.
        2. **Detailed Markdown Report:** Compose a comprehensive, well-structured report in markdown format (at least 500 words) that covers key findings, context, implications, and supporting evidence.
        3. **Follow-up Research Questions:** Suggest 3-5 thoughtful follow-up questions for further investigation.

        ## Input
        - The original user query.
        - Summaries and analyses from the helper tools.

        ## Output
        Return a structured object with:
        - `short_summary`: The executive summary.
        - `markdown_report`: The detailed markdown report.
        - `follow_up_questions`: A list of 3-5 follow-up research questions.
        """,
    model = gpt_model,
    output_type = FinalReport,
)

print("✅ Writer AI Agent is now ready")

✅ Writer AI Agent is now ready


### Build Manager - coordinate workflow of AI agents
**Orchestrate the full pipeline - workflow of the Researcher, Analyst, and Writer agents**

In [22]:
# Create a session to store and share information between agents
session = SQLiteSession("research_agent_practice")

# Define an async function that takes the user's question and runs the agents
async def manager_run(user_query: str):
    # Show the user’s question in a nice markdown format
    print_markdown(f"**User's Request:** {user_query}")

    # Step 1: Run the researcher agent to gather information about the user's query
    researcher_result = await Runner.run(researcher_agent, user_query, session = session)
    research_summary = researcher_result.final_output.summary  # Get the short research summary

    # Step 2: Run the analyst agent to analyze the research findings
    analyst_result = await Runner.run(analyst_agent, research_summary, session = session)
    analysis_summary = analyst_result.final_output.summary  # Get the short analysis summary

    # Step 3: Prepare the combined input for the writer agent
    input_for_writer = (
        f"Original query: {user_query}\n"
        f"Research summary: {research_summary}\n"
        f"Analysis summary: {analysis_summary}"
    )

    # Step 4: Run the writer agent to create the final report
    writer_result = await Runner.run(writer_agent, input_for_writer, session = session)
    final_output: FinalReport = writer_result.final_output  # The final report from the writer

    # Step 5: Display the results in a nice, clear format
    print_markdown("---")
    print_markdown(f"### 📝 **Short Summary:**\n{final_output.short_summary}")  # Quick overview
    print_markdown("\n\n-----------------\n\n")
    print_markdown(f"### 📄 **Full Report (Markdown):**\n{final_output.markdown_report}")  # Detailed report
    print_markdown("\n\n-----------------\n\n")
    print_markdown(
        "### 🔍 **Follow-Up Questions:**\n- "
        + "\n- ".join(final_output.follow_up_questions)  # Any extra questions the user might explore
    )
    print_markdown("\n\n-----------------\n\n")

In [18]:
await manager_run("What is the public sentiment and expert reviews about iPhone 18 Pro and Duo?")

**User's Request:** What is the public sentiment and expert reviews about iPhone 18 Pro and Duo?

- (VIDEO) iPhone 18 Pro and Pro Max Release Date: Expected September 2026 Launch with Major Upgrades: Design rumors point to a refined look with thinner bezels, a cleaner rear camera module and possibly under-display Face ID on the Pro models — eliminating the Dynamic Island notch entirely. Apple is reportedly testing a "deep red" color option for the iPhone 18 Pro, alongside traditional blacks, whites and natural titanium finishes. The all-aluminum unibody introduced in the iPhone 17 Pro series carries over, offering more color flexibility and durability. [...] Pricing is expected to remain consistent with recent Pro models, starting around $1,199 for the iPhone 18 Pro and $1,399 for the Pro Max in the U.S., though storage upgrades and potential feature premiums could push higher-end configurations higher. In markets like India, the iPhone 18 Pro is rumored to start near Rs 1,34,900, with the Pro Max closer to Rs 1,54,900. [...] Camera advancements include a 48-megapixel main sensor with variable aperture control, allowing users to adjust depth of field and light intake dynamically — a first for iPhone. This feature enhances low-light performance, portrait modes and creative control, building on the iPhone 17 Pro's computational photography. Additional upgrades may involve improved ultrawide and telephoto lenses, better stabilization and enhanced AI processing for features like real-time editing and object removal.
- (VIDEO) iPhone 18 Pro and Pro Max Release Date: Expected September 2026 Launch with Major Upgrades: Design rumors point to a refined look with thinner bezels, a cleaner rear camera module and possibly under-display Face ID on the Pro models — eliminating the Dynamic Island notch entirely. Apple is reportedly testing a "deep red" color option for the iPhone 18 Pro, alongside traditional blacks, whites and natural titanium finishes. The all-aluminum unibody introduced in the iPhone 17 Pro series carries over, offering more color flexibility and durability. [...] Pricing is expected to remain consistent with recent Pro models, starting around $1,199 for the iPhone 18 Pro and $1,399 for the Pro Max in the U.S., though storage upgrades and potential feature premiums could push higher-end configurations higher. In markets like India, the iPhone 18 Pro is rumored to start near Rs 1,34,900, with the Pro Max closer to Rs 1,54,900. [...] Camera advancements include a 48-megapixel main sensor with variable aperture control, allowing users to adjust depth of field and light intake dynamically — a first for iPhone. This feature enhances low-light performance, portrait modes and creative control, building on the iPhone 17 Pro's computational photography. Additional upgrades may involve improved ultrawide and telephoto lenses, better stabilization and enhanced AI processing for features like real-time editing and object removal.
- Apple iPhone 18 Rumors: Pro, A20 and Foldable Preview | TechInsights: Copyright © 2026 TechInsights Inc. All rights reserved.

Accessibility | Privacy Policy | Terms & Conditions | Conflict of Interest & Confidential Information | Ethics & Compliance

Upcoming Event   Starts in 

00 Days

:

00 Hrs

:

00 Min

CMC Seminar in Singapore   October 27, 2026 • 8:45 a.m.

Save Your Spot

Join TechInsights  What you’ll learn [...] Apple is expected to announce the iPhone 18 Pro, iPhone 18 Pro Max, and a new foldable iPhone on September 9, 2026.

Reported developments include an A20 Pro processor manufactured using TSMC’s 2 nm process, a possible Apple-designed C2 modem, camera and display upgrades, increased memory, and new foldable display and packaging technologies.

These details remain unconfirmed until Apple’s announcement.

#### Apple iPhone 18 Release Date and Launch Strategy [...] Apple has traditionally introduced its main iPhone lineup together in September. This year could be different.

Reports suggest Apple may divide the iPhone 18 family across two release periods. The iPhone 18 Pro, iPhone 18 Pro Max, and a potential foldable iPhone could arrive in September 2026. The standard iPhone 18, iPhone 18e, and second-generation iPhone Air may follow in spring 2027.
- When To Expect the iPhone 18: A History of Apple’s iPhone Release Dates | WhistleOut: ## iPhone release date: FAQ

### Will there be an iPhone 18?

Yes, the iPhone 18 is expected to be released in March of 2027.

### Is there going to be an iPhone Air 2?

While it's possible the model may be discontinued, most rumors suggest Apple will release the iPhone Air 2 in the spring of 2027, along with the iPhone 18 and iPhone 18e. The fall 2026 release window was dedicated to the iPhone 18 Pro models and Apple's new iPhone Duo.

### When was the iPhone 17 released? [...] Rumors about this year's iPhone 18 series suggesting that Apple would split its newest series into two different events turned out to be true. The iPhone 18 Pro models will be released September 18, and the new iPhone Duo will come out later on October 23. The base-model iPhone 18 has been delayed to spring 2027. [...] The iPhone 18 Pro and 18 Pro Max will come out on September 18, 2026, with preorders beginning on September 12. Preorders for the iPhone Duo begin October 16, 2026, and the device will be officially released on October 23.

iPhones are usually announced at Apple Events in September, and this year's event was no different. This year's event was called "Surprise and Shine" and took place on September 9. This aligned with previous iPhone releases, which typically occur each fall.
- Review: The iPhone 18 Pro is Apple's coolest smartphone (but only literally) - Ars Technica: As I wrote when the Dynamic Island was first introduced, the way Apple combines the per-pixel brightness and absolute blacks of the OLED panel with creative software to turn what should be something of an albatross—a big, chunky hole in the screen—into an appealing feature by displaying live activities and other information around the sensors is one of the company’s more impressive design ideas in recent years. The execution is excellent and remains so. [...] Of course, not too much has changed since the iPhone 17 Pro. There’s the new variable aperture and other camera controls for those who want to get granular with their mobile photography, a welcome addition given the iPhone’s reputation for inflexibly opinionated computational photography. [...] The chip has been re-architected in a couple of ways. New chip packaging promises improved heat management, thanks in part to a design that puts the system-on-a-chip next to the memory instead of stacked on top of it.

- Apple Event 2026 Everything Announced: iPhone Duo, iPhone 18 Pro, Apple Watch 12 and More - CNET: It was first nicknamed the iPhone Fold, and then it was going to be named the iPhone Ultra. Now, new rumors just hours before Apple’s Sept. 9 Surprise and Shine event say that Apple’s first foldable phone will be called the iPhone Duo. Look out, Microsoft — your Surface Duo (a double-screened phone that wasn’t quite a foldable) is about to be properly supplanted. [...] After years of rumors and patience, the iPhone Ultra or iPhone Fold is a highly anticipated addition to Apple’s iPhone lineup and could give other foldable phones like the Galaxy Z Fold 8 series and Pixel 11 Pro Fold some serious competition. [...] Rumors suggest Apple could work the iPhone Air’s sleek design into a book-style foldable iPhone.

Apple’s super-slim iPhone Air wasn’t a blockbuster when it debuted last year, but it still may be one of the company’s most important products. The 5.6mm thick handset could lay the groundwork for Apple’s highly anticipated foldable iPhone, which has speculatively been dubbed the iPhone Ultra.
- Apple event 2026 recap: iPhone Duo, iPhone 18 Pro, and more: Craig Federighi, senior VP of Software Engineering, said that Apple had redesigned much of its iOS software to take advantage of the two screens, moving buttons and other controls. Users can multitask with two apps at the same time and can use Apple's stylus called Pencil.

"We looked at every aspect of iOS to see how we could make it even more useful on iPhone Duo," Federighi said.

—Kif Leswing

It's official, Apple is making a foldable phone. [...] Apple just unveiled the C2 modem, which will power connectivity inside the folding iPhone Duo.

This is the second-generation modem Apple has made in-house, in a move that further reduces reliance on cellular modem leader Qualcomm. [...] "For a device that plays such a central role in your life, a larger display would open up entirely new possibilities," Ternus said. "We think the best way to achieve that is with a foldable design."

He said that other foldables feel like "two phones awkwardly stuck together."

Apple's phone is about the size of a passport and has an internal and external screen.

— Kif Leswing
- Apple Event 2026: Everything announced, including iPhone Duo | Mashable: iPhone Duo runs on the same A20 Pro chip as the Pro models and includes dual batteries that Apple says work together to deliver up to 44 hours of video playback using the outer display alone. The folding form factor enables several camera-specific features, including Smart Take, which uses on-device AI to automatically capture a photo when it detects subjects posing, and Duo FaceTime, which lets a nearby person join a video call through the outer display and camera. [...] It's been awhile since Apple launched a phone that genuinely excited the growing audience of tech skeptics. To wrap the keynote, Ternus introduced iPhone Duo, Apple's first foldable phone. Ternus and company are very confident about Duo, taking shots at the likes of Samsung and Motorola, contrasting it with their foldables as "two phones stuck together." [...] When opened, the Duo has a 7.6-inch inner display and, according to Apple, is the thinnest iPhone ever made; when closed, a 5.4-inch outer display offers 90 percent of the iPhone 18 Pro's screen area. Both screens share the same aspect ratio, so content scales consistently between them, and a nano-texture finish is designed to cut glare and minimize the visibility of the fold crease.
- We Put Apple's iPhone Duo Through HELL. Here’s What Happened.: # We Put Apple's iPhone Duo Through HELL. Here’s What Happened.
## Channel: Joanna Stern (verified)
236K subscribers
11K likes

### Description
551,035 views
Posted: 2026-10-21
It’s here: Apple’s first foldable iPhone. In my iPhone Duo review, I put the folding phone through seven days and seven nights of crazy testing—its hinge, cameras, matte screen and, yes, its software.
- Apple event: CEO John Ternus reveals foldable iPhone Duo | CNN Business: Apple CEO John Ternus holds the iPhone Duo during Apple’s “Surprise and Shine” event at the company's corporate headquarters, Apple Park, on September 9, 2026 in Cupertino, California. The event featured Apple product announcements and was Apple CEO John Ternus’s first keynote address in his new role.

## Today's highlights from Apple event

• Apple unveiled its first foldable phone, the iPhone Duo, marking the biggest change to the iPhone in its nearly 20-year history. [...] Apple Watch Series 12: Starts at $399, up to $1,249.

Apple Watch Ultra 4: Starts at $799, topping out at $899.

## The iPhone Duo's outer display is meant to act like a traditional iPhone screen

Ramishah Maruf
Screenshot 2026-09-09 at 2.41.24 PM copy.jpg

The outer display of the iPhone Duo has 5.4-inch screen and provides 90% of the screen area of iPhone 18 Pro. It will also soon be compatible with the Apple Pencil. [...] The new iPhone Duo foldable offering could account for 10% of Apple’s total iPhone revenue next year, Gene Munster, managing partner of investment firm Deepwater Asset Management, predicted in an X post following the event.

“The new foldable iPhone Duo is sick. No seam. Ultra thin. 50% larger screen than Pro Max,” he said. “My take: They’re going to sell more than I thought before seeing it.”

- Apple iPhone Fold: $1,999 Price, Specs, September 2026 Launch: On April 6, 2026, Chinese leaker Instant Digital revealed that Foxconn had begun trial production of the iPhone Fold at its Zhengzhou facility, a critical milestone in Apple’s manufacturing pipeline. Trial production – known internally as PVT (Production Validation Testing) – typically precedes mass production by 8 to 12 weeks, and DigiTimes reported that same month, in coverage later syndicated by MacRumors, that the mass production ramp-up had slipped from June to early August 2026, a [...] Based on supply chain leaks and analyst reports from 9to5Mac, MacRumors, and Tom’s Guide, the iPhone Fold’s hardware specifications are taking shape. The device features a 7.76 to 7.8-inch inner OLED foldable display manufactured by Samsung Display with a resolution of 1920 x 2713 pixels, 120Hz ProMotion refresh rate, 428 PPI, HDR10+, Dolby Vision support, and 2,000 nits peak brightness. When folded, an outer OLED display serves as the everyday interface – MacRumors’ most recent leak, dated [...] Bloomberg’s Mark Gurman’s prediction held up: Apple unveiled the device – officially named the iPhone Duo – in September 2026 alongside the iPhone 18 Pro and iPhone 18 Pro Max, with Reuters and The New York Times confirming a $1,999 starting price and an October 23, 2026 on-sale date. Trial production began at Foxconn in April 2026, with mass production scheduled for July 2026, though Gadgets Now reported in June 2026 that analysts were already bracing for meaningful volume shipments to slide
- New but shaky report adds to claims the iPhone Fold will launch in September: The likelihood of Apple launching an iPhone Fold in September 2026 seems to change depending on the day. The latest rotating rumors have been either that it's delayed until 2027, or that it is exactly on schedule. Now according to China Securities Journal, a supplier has revealed that Apple has begun shipping the iPhone Fold in small batches. [...] If correct, that would be a significant step considering that the iPhone Fold is believed to have only entered manufacturing testing in April 2026. It's also variously been reported that this testing has uncovered serious problems with the hinge, and also the main circuit board.

There's no information concerning what components this supply chain source produces. But the claim is that this supplier is also relaying the guidance Apple has issued about launching in September 2026. [...] It does appear as if there are more and stronger rumors about the iPhone Fold in 2026 than ever before, though. There have also been dummy models, although those could easily have been made from CAD drawings that themselves could readily have been faked.

But then there are also reports from more reliable sources that, for instance, Samsung has the exclusive contract to make the iPhone Fold's screens.
- Apple Event Today: iPhone 18 Pro and iPhone Duo Cheat Sheet - MacRumors: iPhone 18 Pro pre-orders could begin on Saturday, September 12, at midnight Pacific Time / 3 a.m. Eastern Time. The potential Saturday opening would avoid September 11, similar to Apple's approach in 2015.

The foldable's schedule is less certain, with some reports of manufacturing challenges potentially delaying a release until the fourth quarter of 2026. Indeed, Gurman's latest report claims the device will go on sale "as early as October."

## When Do iPhone 18 Pro and iPhone Duo Launch? [...] The regular iPhone 18 is not expected to feature tomorrow. Apple is reportedly moving that model's launch to spring 2027, when the iPhone 18e and iPhone Air 2 are also expected to debut as part of a new split-year release schedule.

## iPhone 18 Pro and iPhone Duo Pricing Expectations

Higher memory costs could push up prices for both Pro models this year. Based on TrendForce's forecasts for the Pro models and Bloomberg's latest reporting on the foldable, the possible U.S. starting prices are: [...] Apple's first folding iPhone, with "iPhone Duo" among the rumored names, with "iPhone Fold" and "iPhone Ultra" remaining outlying possibilities.
 Book-style design with an approximately 5.5-inch outer display and 7.8-inch inner display.
 Chassis measuring just 4.5mm thick when unfolded.
 Liquid-metal hinge and an extremely inconspicuous – but not completely invisible – display crease.
 Touch ID integrated into the Side button in lieu of Face ID.
- iPhone 18: Rumors and Release Date: Skip to Content

Mac RumorsMac Rumors

Got a tip for us?Let us know

 a. Send us an email
 b. Anonymous form

# iPhone 18

## Apple's iPhone 18 Pro and Pro Max launched in September 2026. Standard iPhone 18 coming in spring 2027.

By MacRumors Staff

iPhone 18

## Ask MacRumors

Powered by AI — responses may not always be accurate

## The iPhone 18 Lineup's Split Launch [...] Apple is deviating from its standard iPhone launch timeline in 2026 and 2027. Instead of introducing four new iPhones in September 2026, the iPhone 18 lineup will come out across fall 2026 and spring 2027.

The more expensive iPhone 18 Pro and iPhone 18 Pro Max launched in September as usual. We got Apple's three most expensive iPhones: iPhone 18 Pro, the iPhone 18 Pro Max, and the iPhone Duo. There was no iPhone 18. [...] ## Launch Date

The iPhone 18, iPhone 18e, and a second-generation iPhone Air are expected to launch in spring 2027.

## iPhone 18 Timeline

October2026

Oct 14Apple Reportedly Planning Large iPhone 18 Production Run

Roundup Last Updated

September2026

Sep 9iPhone 18 Pro Could Charge 25% Faster

Sep 8iPhone Air 2 Could Replace iPhone 18, Sketchy Rumor Claims

Sep 5iPhone 18 Pro: Everything M6 Tells Us About the A20 Chip

August2026
- iPhone 18 Pro and iPhone 18 Pro Max Now Available to Pre-Order - MacRumors: The ‌iPhone 18 Pro‌ and ‌iPhone 18 Pro‌ Max are Apple's latest flagship iPhones, featuring a new variable aperture Main camera, a faster A20 Pro chip, improved vapor chamber cooling, upgraded 5G modems, longer battery life, faster charging, and a smaller Dynamic Island. The new iPhones come in silver, black, glacier, and burgundy. [...] S10 chip, Sound Glow top, Siri AI, and more.

homeview square sidebar

Apple HomeView

Pre-Order Now, Launches Nov 13

Apple's smart home hub with 7-inch square display, A18 chip with Siri AI support, and more.

Apple TV 2022 Feature Blue

Apple TV

Pre-Order Now, Launches Nov 13

A19 chip, Siri AI support, and more.

macbook pro ultra sidebar square

Touchscreen MacBook Pro

October 2026

Apple's first touchscreen Mac, featuring an OLED display, Dynamic Island, and a lighter design.

- Apple's 2026 iPhone Fold Rumors: Crease-Free Design, Price, Launch Date and More - MacRumors: Following some recent speculation that Apple could release its foldable phone under the "iPhone Duo" name, it appears that will indeed be the case, according to Bloomberg's Mark Gurman. For months now, the foldable iPhone has generally been referred to as the "iPhone Ultra," even internally, but Apple is apparently going in a different direction for the official name, emphasizing the...

Read Full Article • 198 comments

iPhone Ultra Fold Feature [...] Apple is expected to unveil its first foldable iPhone during the company's "Surprise and Shine" event on Wednesday, September 9. Reportedly dubbed the iPhone Duo, the device will come in two colors, according to the latest from Bloomberg's Mark Gurman. Previous rumors suggested Apple would "stay away from fun colors" for its first foldable iPhone, with a darker color coming that was...

Read Full Article • 0 comments

apple pencil pro purple [...] by Juli Clover

Apple has been working on a foldable iPhone for multiple years now, and we've been hearing rumors for more than a decade. Information is ramping up, and it looks like we're going to see Apple introduce its first foldable iPhone in September 2026. The foldable iPhone will launch alongside the iPhone 18 Pro models.

Subscribe to the MacRumors YouTube channel for more videos.

  

## What Will Apple's Foldable iPhone Look Like?
- Apple iPhone Fold: $1,999 Price, Specs, September 2026 Launch: Bloomberg’s Mark Gurman’s prediction held up: Apple unveiled the device – officially named the iPhone Duo – in September 2026 alongside the iPhone 18 Pro and iPhone 18 Pro Max, with Reuters and The New York Times confirming a $1,999 starting price and an October 23, 2026 on-sale date. Trial production began at Foxconn in April 2026, with mass production scheduled for July 2026, though Gadgets Now reported in June 2026 that analysts were already bracing for meaningful volume shipments to slide [...] Apple’s first foldable iPhone has entered trial production at Foxconn, and the wait is now over: Reuters and The New York Times confirmed that Apple unveiled the device – officially branded the iPhone Duo – in September 2026 at a starting price of $1,999, with sales opening October 23, 2026, alongside the iPhone 18 Pro and iPhone 18 Pro Max. That launch window matches what Engadget reported back in January 2026, when it found most analysts already converging on a second-half-2026 debut for [...] pairing a roughly 5.5-inch outer display with the roughly 7.8-inch inner screen. With the A20 Pro chip, the iPhone Fold – now officially known as the iPhone Duo – represents Apple’s boldest hardware gamble since the original iPhone – and a direct assault on Samsung’s seven-year dominance of the $31 billion foldable smartphone market.
- The name of the foldable iPhone (2026) | MacRumors Forums: #### wolfedude88

##### macrumors 6502

 #22

iPhone Duo

J

#### JayR

##### macrumors newbie

 #23

iPhone Air Duo

Ted_Appleseed

#### Ted\_Appleseed

##### macrumors regular

Original poster

 #24

> Sami13496 said:
>
> Apple has started naming its new products “Apple Something.” I’m guessing they’ll do the same with Flop.
>
> Click to expand... [...] #8

If I had to choose only one then iPhone Duo. But otherwise it could be anything among Duo/Ultra/Fold.

Reactions: Zwhaler

The-Real-Deal82

#### The-Real-Deal82

##### macrumors P6

 #9

I would be surprised if they called in the Ultra seeing as all rumours suggest it is feature compromised. Calling it 'Fold' is also unlikely as Samsung have that naming convention already and Apple try and steer clear of copying terminology.

Reactions: boomom999, \_Mitchan1999 and Zwhaler

bobtamboo [...] bobtamboo

#### bobtamboo

##### macrumors regular

 #10

I like iPhone duo for a few reasons  
 - for 2 screens  
 - position it as iOS on outer screen, 'iPadOS' features on inner screen (duoOS?)  
 - shouldn't be ultra because some features less than pro / pro max (cameras, faceID etc)  
 - duo price

Reactions: the future and Zwhaler

S

#### Sami13496

##### Suspended

 #11

Apple has started naming its new products “Apple Something.” I’m guessing they’ll do the same with Flop.
- Rumor says foldable is iPhone Duo, gets Apple Pencil, & starts at $2,000 — AppleInsider Forums: This last-minute set of rumors seems rather definitive, suggesting insider information provided the last push of confidence. The details provided are much of what was expected, with just a few surprises.  
  
The iPhone Duo, what was previously called iPhone Ultra or iPhone Fold, will debut during the Apple Event later Wednesday. It will be revealed alongside the iPhone 18 Pro, AirPods 5, Apple Watch Series 12, and Apple Watch Ultra 4. [...] It ain't so. Wrong on nearly every account except the name being similar. It was the Microsoft Surface Duo, this is iPhone Duo, which is also a callback to PowerBook Duo.

  Microsoft's device was two separate displays connected by two hinges suspending the displays separate from each other. There was no external display. [...] Apple wanted a sub-$2,000 price point for its first foldable iPhone, but the AI-driven RAM shortages and other supply chain constraints pushed prices higher. While some costs can be eaten through Apple's usual means, some had to be passed to customers.  
  
According to a report from Bloomberg, Apple's foldable will be called iPhone Duo and start at $2,000 with 256GB of storage. Higher storage tiers will be much more costly to subsidize the entry price.
- iPhone Fold Just Got Delayed And That Changes Everything for iPhone 18 Pro Max!: ### Description
3,837 views
Posted: 2026-03-20
The iPhone Fold might not ship until December 2026 — months after the iPhone 18 Pro hits shelves in September. Barclays analyst Tim Long says Apple's first foldable iPhone could follow a staggered rollout, echoing the same strategy Apple used with the iPhone X in 2017. But this isn't just a production delay. It reveals a deeper shift in how Apple is structuring its entire iPhone release calendar — and most people are reading it wrong. [...] [1:21] first I thought, okay, another delay rumor. A note from Barklay's analyst Tim Long, viewed by Mc Rumors, suggests the
[1:30] iPhone Fold will release in December 2026. He offers no detail as to why it
[1:37] would come out three months after its announcement other than supply chain sources. That's it. That's the detail.
[1:44] Supply chain sources. Three words doing a lot of heavy lifting. That's like saying sources say it'll rain without telling you whether they check the
- Foldable iPhone Duo Will Start at $2,000, Won't Launch Until at Least October - MacRumors: Skip to Content

Mac RumorsMac Rumors

Got a tip for us?Let us know

 a. Send us an email
 b. Anonymous form

# Foldable iPhone Duo Will Start at $2,000, Won't Launch Until at Least October

by Eric Slivka

Following some recent speculation that Apple could release its foldable phone under the "iPhone Duo" name, it appears that will indeed be the case, according to Bloomberg's Mark Gurman. [...] iPhone Ultra Fold Feature  
For months now, the foldable iPhone has generally been referred to as the "iPhone Ultra," even internally, but Apple is apparently going in a different direction for the official name, emphasizing the dual-screen nature of the device. [...] 7 weeks ago

I don’t think it will be called the iPhone Duo. For a company like Apple, especially with the rumored launch approaching, you’d expect them to secure domain names related to the product name well in advance. However, if you visit iphoneduo.com (' it currently redirects to an adult website. That makes “iPhone Duo” seem like an unlikely name.

Score: 15 Votes (Like | Disagree)

adomadosado Avatar

adomadosado

7 weeks ago

---

... (2 outputs left out)

### 📄 **Full Report (Markdown):**
## iPhone 18 Pro and the rumored “iPhone Duo”: sentiment and expert coverage

### Executive perspective

As of March 2026, the iPhone 18 Pro and the foldable sometimes referred to as “iPhone Duo” are subjects of rumors, not products with established review records. Apple has not confirmed the devices or the Duo name. Accordingly, comments about them should be treated as reactions to reported possibilities—not as evidence of how owners or reviewers assess the finished phones.

### What is being reported about the iPhone 18 Pro

Rumor coverage from outlets including MacRumors and Macworld has discussed potential changes to the Pro models, such as a smaller Dynamic Island, a newer-generation chip, and a variable-aperture main camera. These possibilities attract attention because they could touch on features consumers notice: screen design, performance, and photography. If a variable aperture were included, for example, it could offer more direct control over how much light reaches the camera sensor and potentially over depth of field. But the existence, final specifications, and practical quality of such a feature have not been established.

The key limitation is that rumor reporting cannot substitute for hands-on testing. It cannot show whether a camera change produces consistently better photos, whether a new chip improves everyday performance in a noticeable way, or whether a smaller display cutout meaningfully changes usability. Nor can it establish battery life, thermal behavior, software reliability, or value relative to earlier models. It is therefore premature to call the 18 Pro a major upgrade, a minor refresh, or a good purchase.

### The “Duo” and foldable-phone speculation

“iPhone Duo” is an unconfirmed name for a rumored Apple foldable; it is not an announced Apple product. Reports generally describe the idea of a book-style foldable, but the design, specifications, availability, and launch timing remain uncertain. The possibility of Apple entering the foldable category is inherently attention-grabbing because it would introduce a different form factor to the iPhone lineup. That creates curiosity about how Apple might adapt its software and services to a larger folding screen.

At the same time, a new form factor raises practical questions that cannot yet be answered: how visible a display crease might be, how well the hinge would hold up over time, how comfortable the device would be to carry, and whether apps would make good use of the expanded screen. Price is another likely consideration for prospective buyers, but no confirmed price should be assumed from rumor coverage. Until a real product is announced and evaluated, interest in the foldable is best understood as interest in the concept rather than proven demand for a particular device.

### Public sentiment: cautious curiosity, not a consensus

The available information supports a restrained characterization: public discussion appears curious, but confidence is limited. The rumored Pro camera and display changes give existing iPhone users topics to watch, while the foldable concept is likely to attract people interested in new smartphone designs. However, there is no representative polling or substantial owner feedback on either device. Online discussion and technology articles can surface hopes and concerns, but they do not reliably measure the views of all potential buyers.

The most useful takeaway is not that sentiment is definitively positive or negative. Rather, people are waiting for specifics. For the Pro, the central question is whether rumored changes will be meaningful in everyday use. For the foldable, the question is whether Apple can make the format durable, useful, and reasonably compelling despite likely trade-offs in cost and form factor.

### What “expert reviews” mean at this stage

There are no hands-on expert reviews of either device as of March 2026. Current expert coverage is reporting or analysis of rumors, not a review verdict. Analysts and journalists can discuss what a rumored feature might mean, but cannot independently verify its presence or performance. A credible review assessment will require confirmed hardware, access to retail-ready devices, and testing under normal use.

Consumers making an upgrade decision should therefore compare products currently available using confirmed specifications, prices, and independent test results. Waiting for the 18 Pro or rumored foldable may be reasonable for someone who can postpone an upgrade, but basing that decision on unconfirmed features or dates carries risk.

### Bottom line

The early mood around the iPhone 18 Pro and the rumored “Duo” is best described as **interested but unproven**. Potential camera and display refinements, along with the prospect of an Apple foldable, have generated discussion; however, there is no review consensus, meaningful owner sentiment, or confirmed product information. A reliable assessment will only be possible after Apple makes an announcement and independent reviewers test the final devices.



-----------------



### 🔍 **Follow-Up Questions:**
- Which iPhone 18 Pro rumors are supported by the most credible reporting, and which remain speculative?
- How would a variable-aperture camera compare with the current iPhone Pro camera system in independent tests?
- What would prospective buyers prioritize in a foldable iPhone: screen size, durability, software, cameras, or price?
- How might Apple’s rumored release timing affect consumers deciding whether to upgrade now or wait?



-----------------



## Writer AI Agent - (Diff. language)
**Writer function that generate output in different language**

**PRACTICE OPPORTUNITY:**
- **Rewrite the `writer` agent’s instructions so that it generates all outputs entirely in different language (Executive Summary, Detailed Markdown Report, and Follow-up Research Questions).**
- **Note: We will test this AI Agent in the next task so stay tuned!**

In [23]:
## Variation to Report generation.
## **Rewrite the `writer` agent’s instructions so that it generates all outputs entirely in Gujarati (Executive Summary, Detailed Markdown Report, and Follow-up Research Questions).**

writer_agent = Agent(
    name = "Writer",
    instructions = """
You are a senior market analyst tasked with generating an executive-level research report **in Gujarati**.

## Context
You will receive:
- The original user query (the research question or topic of interest).
- Summaries and analyses generated by helper tools: 'researcher_agent' (for bullet-point research findings) and 'analyst_agent' (for key trends, risks, or insights).

You may call these helper tools as needed to gather additional information or clarify findings.

## Instructions
Your job is to synthesize all available information and produce the following outputs in **Gujarati**:
1. **Executive Summary:** Write a concise summary (2-3 sentences) in Gujarati highlighting the most important findings relevant to the original query.
2. **Detailed Markdown Report:** Compose a comprehensive, well-structured report in Gujarati in markdown format (at least 500 words) that covers key findings, context, implications, and supporting evidence.
3. **Follow-up Research Questions:** Suggest 3-5 thoughtful follow-up questions in Gujarati for further investigation.

## Input
- The original user query.
- Summaries and analyses from the helper tools.

## Output
Return a structured object with:
- `short_summary`: The executive summary in Gujarati.
- `markdown_report`: The detailed markdown report in Gujarati.
- `follow_up_questions`: A list of 3-5 follow-up research questions in Gujarati.
""",
    model = "gpt-4.1-mini",
    output_type = FinalReport)


In [24]:
await manager_run("What is the public sentiment and expert reviews about iOS27?")

**User's Request:** What is the public sentiment and expert reviews about iOS27?

- iOS 27 - Hard To Believe!: [22:07] here with iOS 27. Of course, with beta 4 and public beta 2, this is a muchneeded update and so much better than before.
[22:16] Apple continues to do a great job with the latest updates. The stability improves and while it's not perfect, they've made huge strides compared to
[22:24] iOS 26. So, I'm very happy to see the rapid improvement going from beta 3 to beta 4 with everything from apps, [...] [19:59] Optimistic search and Siri finished after 24 hours. Phone is very fast and smooth and battery life so far is very good. had an issue with not being able
[20:08] to make calls, had to reset network settings, and occasionally have issues where, for example, the internet will not connect unless I toggle Wi-Fi off
[20:16] and back on again. Techpro 1 234x3x said, "I'm on iOS 27 public beta 2 on my iPhone 16e. The battery life is really [...] ### Chapters
[0:00] Everything New
[0:41] A New Switch
[1:10] Another Place Added Soon
[1:36] Satellite in more places
[2:03] Updated Protection Options
[2:21] Patched By Apple
[2:36] New Features, Changes and Updates
[4:35] Visual Changes
[5:24] App Updates
[7:00] Apple Releases
[7:16] iOS 26.6 Public Release Date
[7:45] iOS 27 Beta 5 and Public Beta 3 Release (Hard To Believe)
[8:00] iOS 27 Public Release
[8:17] iPhone 18 Pro
[8:43] AirPods Pro 3 Experience
[9:16] iOS 27 Experience
- iOS 27 Cheat Sheet: Quickly Become an Expert on the New iPhone Update - CNET: From new features to updates, here’s everything you should know about iOS 27.

Senior writer

#### Zachary McAuliffe

Senior writer

October 23, 2026, 7:00 am ET

A hand holding an iPhone with 27 on the screen.

The iPhone Duo is out, and it’s running on Apple’s newest operating system, iOS 27.
- iOS 27 Review - What’s New?: [1:50] So, definitely uh very nice there that you can make it more clear or less clear depending on your personal preferences.
[1:57] So, iOS 27 definitely very visually pleasing, very detailed. Uh, it's going to be a really enjoyable software for
[2:05] those of you who really like polish and that's why I think it's great that they launched a duo with this as well as the 18 Pro Max. Now, these are just going to [...] [2:36] 30% faster app launches and 70% faster photo loads. And I can confirm that definitely feels like that. Uh some
[2:44] people you would have you would have to look at it side by side with iOS 26.6, but definitely you can feel that it's very snappy to launch things and not
[2:53] that the iOS wasn't before, but it just seems to be running along the software incredibly fluid. So, so if you like [...] [16:07] pretty strong here uh for iOS 27 throughout the experience. A couple of updates were not the best, but uh generally it's been pretty decent.
[16:15] Hasn't been a horrible experience. So, we got a 3196 on the single core and 8815 on the multi-core. So, pretty
[16:23] respectable scores here for this 17 Pro on iOS 27. So, in conclusion, should you
- With iOS 27, Apple Is Quietly Setting the Stage to Paywall Your iPhone: 5.0 - Exemplary: Near perfection, ground-breaking
 4.5 - Outstanding: Best in class, acts as a benchmark for measuring competitors
 4.0 - Excellent: A performance, feature, or value leader in its class, with few shortfalls
 3.5 - Good: Does what the product should do, and does so better than many competitors
 3.0 - Average: Does what the product should do, and sits in the middle of the pack
 2.5 - Fair: We have some reservations, buy with caution [...] Read Our Editorial Mission Statement and Testing Methodologies.

### About Our Expert

Gabriel Zamora

### Gabriel Zamora

#### Senior Writer, Software

In 2014, I began my career at PCMag as a freelancer. That blossomed into a full-time position in 2021, and I now review email marketing apps, mobile operating systems, web hosting services, streaming music platforms, and video games as a senior writer. I'm a graduate of Hunter College, a hard-core gamer, and an Apple enthusiast. [...] Once I got past the gate, I put the new Siri AI to the test. I wanted to push its limits, so I initiated a complex brainstorming session about a steam-powered airship concept. To my genuine surprise, Siri was incredible: The fluid, natural back-and-forth resembled an advanced generative AI chatbot workspace. It is, without a doubt, a fantastic and genuinely useful tool to have built into a smartphone.
- iOS 27 Is Here. These Are the Best New Features, Including a Smarter Siri. | Reviews by Wirecutter: In iOS 27, Visual Intelligence calculates what each person owes for a restaurant bill from a photo of a receipt, after which they can use Apple Cash to send it to whoever’s covering the bill.

To access the feature, open the Camera app, tap the Visual Intelligence icon, and take a photo of the receipt. The feature calculates what each person owes (including tax and tip), and then each person in the party can select the items and send the total amount owed via Apple Cash. [...] With iOS 27, the photo-editing Clean Up tool is faster and better. You can now choose from different Clean Up modes: Fast (for minor touch-ups using on-device AI models), High Quality (for more complex retouching using Apple’s cloud models), and Auto (in which your device chooses the mode based on what you’re removing). [...] Though app widgets on the home screen are great for viewing more information at a glance, I’d always find myself tapping them to see more details. With iOS 27, you can now make widgets extra large on your home screen. This presentation is useful for apps such as Calendar, News, and Reminders, where you want a more detailed overview.
- iOS 27 Review: Little things mean a lot – Six Colors: It’s at times unsettling to realize this is the new normal for dealing with computers in 2026. Is that better or worse than having old Siri’s frequent habit of shrugging at things it simply couldn’t do? I’m not sure that’s an easy qualitative call. In the end, it’s a mode shift: a different experience of how we interact with our voice assistants and chatbots, and we’ll have to see how people adapt. [...] Dan Moren is the East Coast Bureau Chief of Six Colors, as well as an author, podcaster, and two-time Jeopardy! champion. You can find him on Mastodon at [@dmoren@zeppelin.flights or reach him by email at dan@sixcolors.com. His next novel, the sci-fi adventure Eternity's Tomb, will be released in November 2026.]

 Report a typo
 Discuss in members Discord (app) [...] Six Colors

Six Colors

Apple, technology, and other stuff

#### This Week's Sponsor

Magic Lasso Adblock:Effortlessly blocks ads, trackers and annoyances on your iPhone, iPad, Mac and Apple TV

### By Dan Moren

#### ■ iOS 27

#### ■ review

# iOS 27 Review: Little things mean a lot

- iOS 27: Release Date, New Features, and Compatible iPhones - MacRumors: ## iOS 27: Release Date

iOS 27 has already been available as a developer beta since June and as a public beta since July, and Apple said that the update will be released for all users with a compatible iPhone model this Monday, September 14. We expect the update to roll out at approximately 10 a.m. Pacific Time / 1 p.m. Eastern Time.

## iOS 27: Compatible iPhones [...] Monday October 19, 2026 10:26 am PDT by Juli Clover

Apple today seeded the release candidate of an upcoming iOS 27.2 update to developers for testing purposes, with the update coming a week after Apple released the fourth iOS 27.2 beta. Apple also seeded the iPadOS 27.2 RC. Registered developers can download the beta updates by opting in under Settings > General > Software Update and selecting the iOS 27 developer beta. The beta was used...

Read Full Article • 86 comments [...] iPadOS 27, macOS 27, watchOS 27, tvOS 27, and visionOS 27 will also be released on Monday, September 14, according to Apple.

## iOS 27: New Features

### Siri AI
- iOS 27 - Wikipedia: 1. ↑ Clover, Juli (September 28, 2026). "Apple Releases iOS 27.0.1 With Face ID Bug Fix". MacRumors. Retrieved September 28, 2026.
2. ↑ "Apple Releases Second iOS 27.2 Beta". MacRumors. September 21, 2026. Retrieved September 21, 2026.
3. ↑ "iOS and iPadOS - Feature Availability". Apple. Archived from the original on September 15, 2012. Retrieved May 6, 2026.
4. ↑ Rossignol, Joe (June 8, 2026). "Apple announces iOS 27". The Verge. Retrieved June 8, 2026. [...] ## Version history

iOS 27 version history

| Version | Build number | Release date | Notes |
| Unsupported: 27.0 | 24A427 | September 18, 2026 | Preinstalled on iPhone 18 Pro and 18 Pro Max |
| 24A8428 |
| 24A437 | September 14, 2026 |
| Latest version: 27.0.1 | 24A446 | September 28, 2026 |
| Preview version: 27.2 beta 2 | 24B5089g | September 21, 2026 |
| Legend:  Unsupported  Supported  Latest version  Preview version  Future version |

## References [...] iOS 27 is the twentieth major release of iOS, Apple's mobile operating system for iPhone. Succeeding iOS 26, it was announced on June 8, 2026 at the Worldwide Developers Conference (WWDC), alongside iPadOS 27, macOS Golden Gate, watchOS 27, visionOS 27, and tvOS 27, and became publicly available on September 14.

## Features

### System features

#### Apple Intelligence
- iOS 27: What’s new, what’s in the latest beta and what’s coming next | Macworld: Credit: Foundry

Apple unveiled iOS 27 at WWDC on June 8, 2026, with the public release following on September 14. But more features are on the way, and the latest beta offers a preview of what’s coming next.

iOS 27 mainly focuses on three key areas: Siri AI, a rebuilt assistant with its own dedicated app; new child safety and parental controls; and system improvements designed to make everyday tasks faster and easier. [...] Apple released the iOS 27.2 release candidate to developers on October 19. This is a prerelease build intended for final testing before the update becomes generally available. We expect the public release of iOS 27.2 on October 26 or 27.

Features documented during iOS 27.2 testing include: [...] It seems that Apple will only use iOS 27.1 for the iPhone Duo’s release as it is currently working on the iOS 27.2 beta.

The iOS 27.2 beta 1 was released on September 16, followed by 27.2 beta 2 on September 21. The public beta of iOS 27.2 arrived on September 22. Developer beta 3, was released October 5. Developer beta 4 arrived on October 13.
- For the Fourth Year in a Row, I Was Right. Here’s When iOS 27 Will Land on Your iPhone - CNET: Apple announced iOS 27 at its Worldwide Developers Conference in June, and the tech giant has since released multiple beta versions of the software. The betas have brought Siri AI and a handful of other features to the iPhones of developers and beta testers, and you can expect many of those capabilities to land on your iPhone on Monday, just as I predicted.

Apple usually releases updates around 10 a.m. PT (1 p.m. ET), so that means your iPhone can get an updated Siri AI and more around then. [...] I predicted the exact date Apple would release iOS 27 before the company said anything.

Senior writer

#### Zachary McAuliffe

Senior writer

September 1, 2026, 8:12 am ET [...] I accurately predicted iOS 26’s release in 2025, iOS 18’s release in 2024 and iOS 17’s release in 2023, too. Following a similar formula, here’s how I predicted iOS 27’s release date before Apple even announced it.

## How my iOS 27 release date math works

Usually, Apple releases the next iOS update soon after the September event, and the company held its “Surprise and shine” event on Wednesday.
- iOS 27 Beta 7 Released - What's New?: # iOS 27 Beta 7 Released - What's New?
## Channel: Brandon Butch (verified)
1.44M subscribers
3.9K likes

### Description
191,703 views
Posted: 2026-08-24
iOS 27 Beta 7 Released - What's New? | iOS 27 Features & Changes

Apple just released iOS 27 Beta 7 for the iPhone and it comes with several more new features, changes, and bug fixes! This Beta update includes changes to the spotlight animation, bug fixes, and more. Enjoy!

iOS 27 Beta 7 Release Notes: [...] [9:28] announced this week. We're expecting the the date of the event to be announced this week. And it's most likely going to be either right here the week of September 7th or the week of the 14th.
[9:37] But most people are assuming, you know, most of the rumors in the industry are saying the week of September 7th, potentially on September 8th because we [...] [8:32] go ahead and talk about that RC because we have to talk about what is next for Apple. So, next week is the final Monday of August, and I think that we could possibly see an iOS 27 beta 8 next week.
[8:44] It is possible. However, I think the most likely scenario here is that Apple actually skips next week entirely and then we get the RC build on the week of
[8:54] September 7th. So, it's either that or we get beta 8 next week on Monday and then we skip until maybe September 14th
- iOS 27 Beta 8 Released - What's New?: # iOS 27 Beta 8 Released - What's New?
## Channel: Brandon Butch (verified)
1.44M subscribers
3.9K likes

### Description
236,906 views
Posted: 2026-08-31
iOS 27 Beta 8 Released - What's New? | iOS 27 Features & Changes

Apple just released iOS 27 Beta 8 for the iPhone and it comes with several more new features, changes, and bug fixes! This Beta update includes minor bug fixes and changes to the home screen, Find My, the keyboard, and more. Enjoy!

iOS 27 Beta 7/8 Release Notes: [...] [9:37] anyways, needless to say, I think we are going to see the RC next right there on Wednesday, September 9th after Apple's big September event where they unveil
[9:45] the new iPhone 18 Pro and Pro Max and the iPhone Ultra or iPhone Fold. So with that being the case, I think we're going
[9:53] to see the final official release of iOS 27 on Monday the 14th. So, Monday, September 14th, exactly two weeks from [...] [10:24] the other Apple devices right there on the 18th. And then of course we do have the Max also releasing the following week, the first half of the week there
[10:32] on the 22nd. So a lot of big things coming up for Apple, but of course we're talking mostly about software here. So that is the latest with the software release dates. So that is iOS 27 beta 8.

- iOS 27 Review: AI Finally Takes Over The iPhone: © 2026 Static Media. All Rights Reserved

BGR

BGR

iOS 27 Review: AI Finally Takes Over The iPhone

 Technology
 Apps & Software

# iOS 27 Review: AI Finally Takes Over The iPhone

By José Adorno

A hand holding an iPhone with the new Siri AI  José Adorno/BGR

Add BGR on Google:

Google Discover [...] Of course, battery life is always a major concern when updates arrive. I've been charging my iPhone twice a day throughout the summer, which is due to a mix of Parisian heatwaves, testing every Siri AI feature, and a seemingly dry vapor chamber (that's not really how it works, but my iPhone 17 Pro Max does feel warmer around the camera plateau). However, for those who haven't been using the iOS 27 beta, Apple has been doing something clever: indexing Siri and its apps for the iOS 27 release. [...] Besides the full adoption of AI, iOS 27 is fundamentally a minor upgrade. There are no notable new features for Apple Music, most of Apple's apps look the same, and highly-anticipated additions, like the ability to update passwords with a single tap, aren't ready yet. Rather, Apple decided to deliver an update focused on fixing bugs, polishing up the code, and making the experience feel snappier.
- iOS 27 Review: Is It ACTUALLY Faster? Plus New Siri & Features: ### Transcript
[0:00] WWDC 2026 was different. No flashy new feature, no redesign. Instead, iOS 27
[0:07] feels like a repeat of the legendary iOS 12 WWDC. It focuses entirely on performance, stability, and polish. The
[0:15] result, app launch is up to 30% faster, airdrop up to 80% faster. Even old iPhones start to feel new again. But
[0:23] then that's just the start. So, what's actually new under the hood? Apple rewrote core components. They fix bugs. [...] # iOS 27 Review: Is It ACTUALLY Faster? Plus New Siri & Features
## Channel: BigBadroid
10.8K subscribers
7 likes

### Description
464 views
Posted: 2026-06-09
iOS 27 is here, and after running the developer beta, I'm breaking down everything you need to know: the new features, performance improvements, and the all-new Siri AI. [...] ⚡ Under-the-Hood Speed & Performance:
iOS 27 is all about polish. Apple has focused on making your iPhone feel faster than ever:
Apps launch up to 30% faster thanks to a new CPU scheduler
Photos load in your library up to 70% quicker
AirDrop transfers are now up to 80% faster
- iOS 27 lands September 14 — but Siri AI is blocked in the EU at launch: iOS 27 rolls out on September 14, 2026, for every compatible iPhone from the iPhone 11 onwards. Apple's biggest software update in years brings a revamped Siri powered by generative AI, but users in the EU — including the UK — won't get that flagship feature at launch because of ongoing regulatory hurdles. Everyone else gets 250-plus new features on day one.

## The new Siri [...] Gagadget
 News
 Apple News
 iOS 27 lands September 14 — but Siri AI is blocked in the EU at launch

# iOS 27 lands September 14 — but Siri AI is blocked in the EU at launch

By: Anton Kratiuk | 10.09.2026, 13:53

iOS 27 lands September 14 — but Siri AI is blocked in the EU at launch [...] ## The new Siri

The centerpiece of iOS 27 is a fully rebuilt Siri that uses generative AI models developed in partnership with Google and Nvidia. It's designed to handle complex, multi-step requests — think booking a restaurant while checking your calendar — rather than just setting timers and playing music. Apple has spent several years on the retraining, and Craig Federighi has publicly expressed frustration that European regulation is holding the rollout back.
- Instagram: krishn\_singh9014

Ha

Reply

divyang.antiya.6064's profile picture

divyang.antiya.6064

Bhai realme gt 7t lena chahiye kya bbd me

Reply

not_noni_'s profile picture

not\_noni\_

Mera to 13 pro h refurbished liya tha february me 2026 or ios 27 bilkul smooth chal ra h koi dikkat nai aa rhi 💝❤️

Reply

rangrejritika's profile picture

rangrejritika

Isliye Maine kiya he nahi 🌚👀

1 like

Reply

rii.riiyaa's profile picture

rii.riiyaa

Nhi krenge install😭

2 likes

Reply [...] iOS 27 bugs | iOS 27 issues | iOS 27 update problems | iPhone update | iOS 27 battery drain | iOS 27 heating issue | iOS 27 review | Apple iOS 27 | iPhone software update  
  
#iOS27 #iPhone #Apple #iOSUpdate #iPhoneTips TechNews TechShorts Shorts [...] ios 27, ios 27 bugs, ios 27 issues, ios 27 problems, ios 27 glitches, ios 27 update, ios 27 review, ios 27 battery drain, ios 27 heating issue, ios 27 lag, ios 27 performance, ios 27 update problems, ios 27 bugs fix, iphone ios 27, iphone update, apple ios 27, ios 27 features, ios 27 hidden bugs, iphone software update, ios update, iphone tips, apple update, iphone news, ios 27 india, iphone tricks, tech news, iphone shorts, ios shorts, tech shorts, shorts
- Apple Needs to Solve These iOS 27 Bugs immediately: ## Battery Drain and Overheating: The Usual Suspects, But Worse This Time

Press enter or click to view image in full size

Image 4 [...] Sitemap

Open in app

Sign up

Sign in

[](

Get app

Write

Search

Sign up

Sign in

Image 1: Unknown user

Member-only story

IOS

Ios 27

Apple

Technology

IOS Development

# Apple Needs to Solve These iOS 27 Bugs immediately

Image 2: apple insides

apple insides

5 min read

·

Oct 3, 2026

[](

--

2

[](

[](

Listen

Share

Press enter or click to view image in full size

Image 3 [...] Image 3

iOS 27 launched on September 14 carrying Apple’s biggest promise in years a completely rebuilt Siri AI, deeper Apple Intelligence integration, and genuine performance polish across the board, according to Rapid Repair.

But less than two weeks later, a genuinely persistent list of bugs has piled up, and some of them go well beyond the usual “give it a few days” first-week jitters.

Here’s everything actually going wrong, and what needs fixing.
- iOS 27 Review: Little things mean a lot – Six Colors: It’s at times unsettling to realize this is the new normal for dealing with computers in 2026. Is that better or worse than having old Siri’s frequent habit of shrugging at things it simply couldn’t do? I’m not sure that’s an easy qualitative call. In the end, it’s a mode shift: a different experience of how we interact with our voice assistants and chatbots, and we’ll have to see how people adapt. [...] The good news is there are some places where things are just better across the board. In general, iOS 27 feels peppy. I know, we always talk about snappiness, but given the time Apple has spent on improving performance and removing bugs with this release, it’s reassuring to see that it’s borne out by my experience. Apps launch quickly, photos sync faster via iCloud, AirDrop transfers are especially rapid, browsing my NAS via the Files app was downright speedy compared to earlier versions. [...] While Apple Intelligence is still apparently a going concern, much of that brand has been subsumed by the “Siri” identity in iOS 27. For example, any time you put a blank cursor in a text field, you’re prompted with a Write with Siri button hovering aboard the keyboard. (Once you start typing, you get the usual word suggestions, and Siri is reduced to just an icon.) As someone who is perfectly happy writing their own text messages and emails, this prompt sometimes feels a little bit aggressive,
- iOS 27 Is Here. These Are the Best New Features, Including a Smarter Siri. | Reviews by Wirecutter: With iOS 27, the photo-editing Clean Up tool is faster and better. You can now choose from different Clean Up modes: Fast (for minor touch-ups using on-device AI models), High Quality (for more complex retouching using Apple’s cloud models), and Auto (in which your device chooses the mode based on what you’re removing). [...] iOS 27 is compatible with the iPhone 11 and newer, but only the iPhone 15 Pro and newer support Apple Intelligence features such as Siri AI, the standalone Siri app, and Visual Intelligence. The following features are limited to those newer phones. To learn about iOS 27 features that are available on older iPhones, jump ahead to that section.

### Rely on a smarter, more capable Siri [...] Despite such hiccups, I rely on Siri a lot more now than I have in years. I used it to plan an itinerary for a day out in Soho, and it recommended various coffee shops, restaurants, and bookstores in the area. I used it to analyze screenshots of clothing items and accessories to find similar items online. I used it for looking up movie times at local theaters and for finding recipes using ingredients I had in my fridge. All of these queries worked without any issues.
- iOS 27 battery life thread. | MacRumors Forums: #24

So about a day later I have seen a few small bugs and my iPhone 17 keeps getting hot/warm. But other than that performance seems faster than iOS 26 especially opening up apps. But it’s still processing/indexing in the background I will give it a few more days. Btw I got the new Siri AI I was off the waitlist after 22 hours.

krspkbl

#### krspkbl

##### macrumors 68030

 #25 [...] #1

Hey everyone, how’s WWDC 2026 treating you? We’ve heard a new iOS is coming out, and it’s time to dive into the iOS 27 battery life thread! It’s still surprising how many iPhone users are holding off on updating beyond iOS 18, especially since the Liquid Glass update drained battery health even with new features like Adaptive Power!  
   
 Want to see a quick rundown of iOS 26 battery life and health? Here are some highlights: [...] > xxFoxtail said:
>
> So far, battery life is a lot worse for my iPhone Air. Warms up a lot more than it did on iOS 26 too.
>
> Click to expand...

  
> lemonviking said:
>
> Weird. I have a m1 ipad pro too and it barely heats up.
>
> Click to expand...

- iOS 27 Problems: Battery Drain, Heat & Siri AI Bugs: Last updated: September 15, 2026. This is a developing first-day report and will be updated as longer battery tests and broader user data become available.

## What iOS 27 problems are users reporting?

ZEERA reviewed early public reports and ran practical checks focused on battery use, temperature, Siri AI activation, update labeling, storage and everyday stability. Because iOS 27 has been public for only about a day, these findings are signals—not measured failure rates.

REPEATED REPORTS [...] Quick Answer

iOS 27 is not showing signs of a widespread, device-breaking failure after its first day, but users are reporting battery drain, extra heat, Siri AI access problems and update-label confusion.

ZEERA’s first-day tracking and hands-on checks suggest that heavier battery use and warmth are most noticeable while the iPhone finishes background work after installation. Give the update 48–72 hours before judging battery life. [...] Japan and South Korea Restrict iPhone Duo and iPhone 18 Pro Max Walk-In Purchases  Oct 20, 2026

iOS 27.2 and iPadOS 27.2 RC Released: What Build 24B115 Means  Oct 20, 2026

Apple Announces “Pro and Tell” for October 28: New Macs and iPad mini Expected  Oct 20, 2026

iPad mini 8 Looks Increasingly Likely to Get 120Hz ProMotion  Oct 20, 2026

Best Apple iPhone Duo Folio Alternative: ZEERA FoldTex Pro for Less  Oct 20, 2026

Fast delivery worldwide

100% Customer satisfaction
- iOS 27 Beta 8 Reddit Discussion: Keyboard, Battery, and ...: iOS 27 Beta 8 brings stability and performance tweaks, but users report keyboard lag, battery drain, and Siri AI bugs. Most agree it's smoother on newer
- IOS 27 Beta : r/iOS27: I have fixed overheating and battery draining by disabling Siri completely. To do this select different siri language in which siri is not
- iOS 27 bugs cause iPhone crashes, battery drain, restarts | Cybernews: “I’m using an iPhone 15 Pro Max (battery health 84%, around 828 cycles). After updating to iOS 27, I’m noticing higher battery drain than before,” wrote one Redditor.

While the user thought the issue might be related to battery health, a number of iPhone owners also noticed their battery life changed after the update. [...] © 2026 Cybernews – Latest Cybersecurity and Tech News, Research & Analysis.

# Crashing UI, drained battery life, and random restarts: iPhone users report bugs after updating to iOS 27

This update will need an update.

iOS 27 software update overview

iOS 27 software update overview. Image by Bangla press | Shutterstock

Konstancija Gasaitytė

Key Takeaways by nexos.ai, reviewed by Cybernews staff. [...] Along with new devices, Apple also released its latest software system, iOS 27, earlier this month. However, some users are unhappy with the upgrade, as it seems riddled with bugs.

A number of users took to social media channels such as Reddit and X to share their impressions after updating their devices to the latest version of Apple's smartphone software.
- Thank you iOS 27! : r/ios: It slowed my phone, caused heating problems, and regular glitching issues. Changed my battery hoping it would become alright. But no, turns out
- iOS 27 Public Beta 4 — Anyone else having terrible battery ...: Is anyone else experiencing really bad battery performance on iOS 27 Public Beta 4?
I’m using an iPhone 15 Pro Max with 85% Battery Health, and since
- Conflicting reports on iOS 27 battery life: I have seen opinions all across the board:

"iOS 27 has improved my battery life and breathed new life into my phone!"

"iOS 27 has absolutely destroyed
- iOS 27 Beta 3 - Discussion : r/iOSBeta: Anyone else having issue with Siri being able to tell you random facts but you ask it to do anything like set a reminder for when I get home,

- iOS 27 Review: Little things mean a lot – Six Colors: New Siri is integrated throughout iOS, whether you access it via the traditional press-and-hold of the side button or by saying the wake word, or in its new home integrated with the Spotlight search when you swipe down on the home screen. That last interface is available systemwide, in any app, by swiping down from the center of the top of the screen. The search interface has a new look too: a black liquid glass bubble with silver overtones. You can ask Siri questions by voice, or by simply [...] The promise of iOS 27 is that it’ll not only unlock new capabilities but also make your existing workflows smoother and quicker. That’s not a bad proposition for the device that many of us carry more than any other—as long as it can deliver.

## Siri & AI: Two okay tastes that taste great together? [...] An exchange with Siri AI about playing a podcast episode.

Play it, Siri. Play As Time Goes By The Rest is History.
- With iOS 27, Apple Is Quietly Setting the Stage to Paywall Your iPhone: PCMag Australia

Looking for a bargain? Check out today's best online tech deals here

PCMag Australia OpinionOperating Systems iOS

# With iOS 27, Apple Is Quietly Setting the Stage to Paywall Your iPhone

## Apple has introduced an excellent conversational AI assistant, but it also exposes an uncomfortable corporate playbook that could soon monetize your mobile.

Gabriel Zamora

& Gabriel Zamora Senior Writer, Software [...] Once I got past the gate, I put the new Siri AI to the test. I wanted to push its limits, so I initiated a complex brainstorming session about a steam-powered airship concept. To my genuine surprise, Siri was incredible: The fluid, natural back-and-forth resembled an advanced generative AI chatbot workspace. It is, without a doubt, a fantastic and genuinely useful tool to have built into a smartphone. [...] 5.0 - Exemplary: Near perfection, ground-breaking
 4.5 - Outstanding: Best in class, acts as a benchmark for measuring competitors
 4.0 - Excellent: A performance, feature, or value leader in its class, with few shortfalls
 3.5 - Good: Does what the product should do, and does so better than many competitors
 3.0 - Average: Does what the product should do, and sits in the middle of the pack
 2.5 - Fair: We have some reservations, buy with caution
- PCMag: Log In

Log In

Forgot Account?

## PCMag's Post

  

[](

### PCMag

September 20 at 2:04 PM · 

Now that the final public release of iOS 27 is broadly available, it's time to see what Apple's latest operating system can do. Start poking around Apple's revamped, contextual Siri, set up the new parental controls, and more with these tips.

Image 1: 🌐 Did You Download iOS 27? Do These 6 Things First

Image 2: 👇 Link below Image 3: 👇

Image 4: Did You Download iOS 27? Do These 6 Things First [...] Image 5

Image 6

All reactions:

23

1 comment

3 shares

Like

Comment

Most relevant

[](

Image 7Author

PCMag

Read article Image 8: 👀bit.ly/46vKdAT

Image 9: Did You Download iOS 27? Do These 6 Things First

PCMAG.COM

Did You Download iOS 27? Do These 6 Things First

### Did You Download iOS 27? Do These 6 Things First

   4w 

1

Image 10

Image 11

See more from PCMag

See more from PCMag

Email or phone number

 

Password

 

Log In

Forgot password?

or

Create new account
- iOS 27 Review: Is It ACTUALLY Faster? Plus New Siri & Features: # iOS 27 Review: Is It ACTUALLY Faster? Plus New Siri & Features
## Channel: BigBadroid
10.8K subscribers
7 likes

### Description
464 views
Posted: 2026-06-09
iOS 27 is here, and after running the developer beta, I'm breaking down everything you need to know: the new features, performance improvements, and the all-new Siri AI. [...] ⚡ Under-the-Hood Speed & Performance:
iOS 27 is all about polish. Apple has focused on making your iPhone feel faster than ever:
Apps launch up to 30% faster thanks to a new CPU scheduler
Photos load in your library up to 70% quicker
AirDrop transfers are now up to 80% faster [...] 🤖 The BIG Upgrade: Siri AI (Hands-On Demo)
The biggest headline is the arrival of Siri AI. It’s more than just a voice assistant; it's a standalone app with a chat interface, capable of:
Understanding context across your apps (Messages, Calendar, etc.)
Recognizing and acting on what's on your screen
Providing natural, conversational answers
Note: Requires iPhone Air, 17 Pro series, or M3/M4 devices for full on-device features
- iOS 27 lands September 14 — but Siri AI is blocked in the EU at launch: ## The new Siri

The centerpiece of iOS 27 is a fully rebuilt Siri that uses generative AI models developed in partnership with Google and Nvidia. It's designed to handle complex, multi-step requests — think booking a restaurant while checking your calendar — rather than just setting timers and playing music. Apple has spent several years on the retraining, and Craig Federighi has publicly expressed frustration that European regulation is holding the rollout back. [...] iOS 27 rolls out on September 14, 2026, for every compatible iPhone from the iPhone 11 onwards. Apple's biggest software update in years brings a revamped Siri powered by generative AI, but users in the EU — including the UK — won't get that flagship feature at launch because of ongoing regulatory hurdles. Everyone else gets 250-plus new features on day one.

## The new Siri [...] The Photos app gains AI-powered editing tools for more precise retouching. The Liquid Glass interface gets deeper customisation, including transparency controls. Security and parental-control features are also expanded across the board.

## The EU gap
- Apple plans to open up Siri to rival AI assistants in iOS 27 update: Load More [...] VIDEO
04:11
Anthropic investor Matt Witheiler: Company's enterprise pivot was 'absolutely' important
watch now
watch now
VIDEO
02:42
Anthropic investment will be 'by far the biggest return' in Menlo's history, says partner Matt Murphy
watch now
watch now
VIDEO
02:25
Anthropic is Bessemer's largest investment in its 116-year history, says early investor Byron Deeter
watch now
watch now
VIDEO
35:54
Why Kunal Kapoor turned down a better-paid job to join Morningstar
5 hours ago
watch now
watch now [...] Power Lunch
Apple plans to open up Siri to rival AI assistants in iOS 27 update
CNBC's MacKenzie Sigalos reports on news regarding Apple.
01:27
Thu, Mar 26 2026
3:06 PM EDT
MacKenzie Sigalos
watch now
watch now
VIDEO
35:54
Why Kunal Kapoor turned down a better-paid job to join Morningstar
5 hours ago
watch now
watch now
VIDEO
01:40
Microsoft offering buyouts to some employees in U.S., Canada, India and U.K.
watch now
watch now
VIDEO
04:11
- Apple to open Siri to third-party AI assistants in iOS 27 - Bloomberg: Plus500

4.9

★★★★★

Open Account

Read Review

Image 2: eToro

eToro

4.8

★★★★★

Open Account

Read Review

   Tools 

Calendars 
       Economic Calendar
       Holiday Calendar
       Earnings Calendar
       Dividend Calendar
       Splits Calendar
       IPO Calendar
       Futures Expiry Calendar

Investment Tools

       Stock Screener
       Fed Rate Monitor Tool
       Currency Converter
       Fibonacci Calculator

More In Tools [...] English (USA)
   English (UK)
   English (India)
   English (Canada)
   English (South Africa)
   English (Philippines)
   English (Nigeria)
   Deutsch
   Español (España)
   Español (México)
   Français
   Italiano
   Nederlands
   Polski
   Português (Portugal)
   Português (Brasil)
   Русский
   Türkçe
   ‏العربية‏
   Ελληνικά
   Svenska
   Suomi
   עברית
   日本語
   한국어
   简体中文
   繁體中文
   Bahasa Indonesia
   Bahasa Melayu
   ไทย
   Tiếng Việt
   हिंदी [...] © 2007-2026 -Fusion Media Limited. All Rights Reserved.

   Terms And Conditions
   Privacy Policy
   Risk Warning

Image 25Image 26Image 27Image 28Image 29Image 30Image 31Image 32Image 33Image 34Image 35Image 36Image 37Image 38Image 39Image 40Image 41Image 42Image 43Image 44Image 45Image 46Image 47Image 48Image 49Image 50Image 51Image 52Image 53Image 54

... (6 outputs left out)

### 📄 **Full Report (Markdown):**
# iOS 27 પર જનપ્રતિભાવ અને નિષ્ણાત સમીક્ષાઓ

iOS 27 એ Appleની ગુજરાતી એપલ ટ્રેડ યુઝર્સ માટે રજૂ કરેલી નવીતમ ઓપરેટિંગ સિસ્ટમ છે. એ માટેની જનપ્રતિભાવ અને નિષ્ણાત સમીક્ષાઓ વિવિધ છે, જેને નીચે વિગતવાર ચર્ચા કરવામાં આવી છે.

## જનપ્રતિભાવ

- કેટલીક વપરાશકર્તાઓ iOS 27 ના ઝડપી અને સરળ ઉપયોગ માટે સારી પ્રતિભાવ આપે છે. એપ્લિકેશનો ઝડપથી ખૂલે છે, ફોટા ઝડપી લોૈડ થાય છે અને AirDrop ટ્રાન્સફર વધુ ઝડપી અનુભવાય છે.
- બીજી બાજુ, કેટલાક વપરાશકર્તાઓ બેટરી ઝડપથી ખતમ થવી, ફોન ગરમ થવું અને કેટલીકવાર એરલેસે ઉપકરણ અચાનક રીસ્ટાર્ટ થવું જેવી સમસ્યાઓના શિકાર બન્યા છે.
- આ ફરિયાદો ખાસ કરીને નવા સુધારાઓ પછી પાછા આવતા બેગ્રાઉન્ડ પ્રોસેસિંગ હોવાથી હોઈ શકે છે, પણ આ સમસ્યાઓ કેટલી પ્રમાણમાં વ્યાપક છે તે હાલ સ્પષ્ટ નથી.

## નિષ્ણાત સમીક્ષાઓ

- Six Colors ના ડેન મોરેન આ સુધારાને મુખ્યત્વે પ્રદર્શન અને સ્થિરતામાં મળતી સુધારાઓ માટે વખાણ કરે છે, પરંતુ Siri દ્વારા આપેલી લખાણ સૂચનાઓ કંઇક વધુ હદ સુધીની લાગણીસભર લાગી હતી.
- Wirecutter ની સમીક્ષા મુજબ, પ્5રયોગી iOS 27 ની નવી Siri ને ઉપયોગી અને સિદ્ધાંતર રીતે બહોળા કાર્યક્ષેત્રોને આવરી લેતી જોઈએ છે, જેમ કે લાઇવ પ્લાનિંગમાં સહાય અને પાકો માટે રેસીપી શોધવી.
- PCMag ના નિષ્ણાતોએ Siriના સંવાદાત્મક ક્ષમતા માટે પ્રશંસા કરી છે, સાથે જ ભવિષ્યમાં તેના પ્રમાણવાર ઉપયોગથી કોઈ પૈસાની આવક માટેની યોજના વિશે ચિંતા વ્યક્ત કરી છે.

## જોખમો અને ચિંતાઓ

- વધુ બેટરી ડ્રેન અને ફોન ગરમ થવું એ iOS 27 સાથે સંકળાયેલા મુખ્ય સમસ્યાઓમાં શામિલ છે.
- કેટલાક યૂઝર્સ માટે ફોનનું અચાનક રીસ્ટાર્ટ થયવાનું અને ક્યારેક કનેક્શન સમસ્યાઓ જોવા મળી રહી છે.
- નવી Siri એ ઈયૂમાં નિયમન સમાચાર અને નિયમો વડે મોડું લૉન્ચ થવાનું કારણ બની છે.

## સંક્ષિપ્ત નિષ્કર્ષ

iOS 27 વપરાશકર્તાઓ માટે ફીટનેસ અને ત્વરિત કામગીરી લઈને આવતો મજબૂત અપડેટ છે, પરંતુ હાલ કેટલીક સલામતી અને બેટરી સંબંધિત સમસ્યાઓ તેમજ Siri ની કેટલીક અચોક્કસતા હજુ પણ યથાવત્ છે. ઉપભોક્તાઓ માટે સુધારાઓ સાથે અપડેટને અપનાવવું શ્રેષ્ઠ રહેશે અને Apple પાસેથી ચાલુ સપોર્ટની અપેક્ષા રાખવી જોઇએ.



-----------------



### 🔍 **Follow-Up Questions:**
- iOS 27 ના ખાસ બેટરી બચત માટેનાં નવા ફીચર્સ શું છે?
- Siri AI ની નવી ક્ષમતા અને તેની સંપૂર્ણ ફંક્શનલિટી કયા ઉપકરણો પર ઉપલબ્ધ છે?
- iOS 27 માં સુરક્ષા અને વિધિ નિયંત્રણના નવા ઉપાય શું પ્રસ્તુત કરાયા છે?
- વિશ્વભરના વપરાશકર્તાઓ દ્વારા હાજર બગ્સ અને તકલીફોનું Apple દ્રારા નિકાલ કેવી રીતે કરાયું છે?
- iOS 27 માંથી નિકળેલા નવીનતમ અલગ પ્રકારના બિલ્ડ અને અપડેટ કોલસલ શું છે?



-----------------



## PRACTICE SOLUTION

- **Use the `researcher_agent` and `analyst_agent` to analyze a specific research question or topic.**

  - **Provide a research question (e.g., "What are the latest trends in Batteries for Electric Vehicles?").**
  - **Call the `researcher_agent` with the research question to obtain key findings.**
  - **Pass the researcher's summary to the `analyst_agent` to extract key trends, risks, or insights.**
  - **Show how you would structure the conversation and pass information between the agents.**


In [25]:
# Let's run the researcher AI agent
researcher_result = await Runner.run(researcher_agent, "Electric vehicle batteries trends in 2026 and beyond")
print_markdown(f"### 🤖 Researcher Agent's Reaserach\n{researcher_result.final_output.summary}")

- Electric vehicle batteries - Global EV Outlook 2026 - IEA: Record low LFP battery prices also contributed significantly to overall battery price reductions in 2025. In 2025, LFP battery packs were more than 40% cheaper on average than lithium nickel manganese cobalt oxide (NMC) alternatives per kWh, although part of this cost advantage reflects the lower energy density requirements of stationary storage applications, which predominantly use LFP3. While LFP batteries benefit from structurally lower material costs, there are growing concerns about the [...] Despite recent progress, sodium‑ion batteries remain constrained by lower energy density than lithium‑ion technologies, limiting their competitiveness with LFP batteries at current lithium prices. The latest sodium‑ion cells can reach up to 175 Wh/kg, compared with up to 205 Wh/kg for the latest generation of LFP batteries and 265 Wh/kg for lithium nickel cobalt manganese oxide (NMC) batteries. Their disadvantage is even greater in terms of volumetric energy density (Wh/l), which translates [...] Average battery prices declined by 8% in 2025, supported by continued improvements in manufacturing efficiency, advances and shifts in battery chemistries and technology, and intensifying global market competition. Relatively low critical mineral prices also contributed to downward cost pressure, although lithium and cobalt experienced notable price increases over the year. The recent increase in lithium and cobalt prices – if sustained – could put upward pressure on battery costs as stockpiles
- Global EV Outlook 2026: Growing sales amid an energy crisis: rise, including faster‑than‑anticipated demand growth – particularly from the battery energy storage sector –, relatively low inventories in China and temporary supply disruptions, such as the suspension of operations at CATL’s Jianxiawo lithium mine. If these upward price trends persist, they could exert upward pressure on lithium-ion battery prices and reinforce the current momentum behind sodium‑ion batteries, which offer lower ranges but do not rely on lithium. This effect could [...] 24 Shares refer to average battery cell production cost in China in 2024. Global EV Outlook 2026 Chapter 5. Trends in electric vehicle batteries PAGE | 101 IEA. CC BY 4.0. batteries at current lithium prices. The latest sodium-ion cells can reach up to 175 Wh/kg, compared with up to 205 Wh/kg for the latest generation of LFP batteries and 265 Wh/kg for lithium nickel cobalt manganese oxide (NMC) batteries. Their disadvantage is even greater in terms of volumetric energy density (Wh/l), which [...] lines represent the sales-weighted average battery pack size for battery electric cars, while dashed lines indicate the same for plug-in hybrid electric cars. Source: IEA analysis based on EV Volumes. Battery industry trends Battery prices continued to decrease, albeit unevenly Average battery prices declined by 8% in 2025, supported by continued improvements in manufacturing efficiency, advances and shifts in battery chemistries and technology, and intensifying global market competition (see
- Global Market for Sodium-ion Batteries 2026-2036: Sodium-Ion Battery Market to Surpass $30B by 2036, Disrupting Lithium Dominance: These systems provide crucial grid stabilization services for renewable energy integration, addressing the intermittency challenges of solar and wind power while offering cost advantages over lithium-ion alternatives. Automotive Applications are rapidly emerging, particularly in the budget EV segment. JAC Motors' pioneering sodium-ion production vehicle, featuring a 25kWh battery with 155-mile range, validates the technology's automotive viability. Industry reports suggest major manufacturers [...] With sodium priced at just $0.05 per kilogram compared to lithium's $15 per kilogram, manufacturers can achieve significant cost reductions while maintaining comparable performance characteristics. This 300-fold price differential in raw materials translates directly into more affordable battery systems, positioning sodium-ion technology as a game-changer for price-sensitive applications. [...] and environmental benefits positions sodium-ion technology as a cornerstone of the global energy transition, promising to democratize access to clean energy storage solutions.
- International Energy Agency (IEA): Image 2: May be an image of text that says 'Electric vehicle battery sales share by chemistry and region, 2023-2025 Global EV Outlook GlobalEVOutlook2026 Outlook202 2026 100% cam 80% • LFP • Low-nickel E Mid- and high- high-nickel ANAN Loni 60% 40% 20% 0% 2023 2024 2025 2023 2024 Source: EMDES= emerging marketsar and developing economies: LFP= Lithiumiron phosphate 2025 2023 2024 2025 2023 2024 2025 International Agency Energy'

Image 3

All reactions:

44

1 comment

16 shares

Like

Comment
- Global EV Outlook 2026 - IEA: Combining analysis of historical data with projections, the report examines key areas of interest, such as the deployment of electric vehicles and charging infrastructure, battery demand, and key policy developments in major and emerging markets. It also considers the implications of growing EV adoption for electricity and oil consumption, as well as greenhouse gas emissions. [...] Key technology trends are aligning in favour of EVs

Electric vehicles are increasingly at the centre of innovation in the automotive sector, with advances that extend beyond developments in batteries and power electronics explored in these key technologies

|  | [...] Amid the ongoing energy crisis sparked by the conflict in the Middle East, this edition includes early monthly data for 2026 and considers potential implications of the crisis for policy and market development. It also includes analysis of the affordability of electric cars and the manufacturing and trade of electric cars, trucks and their batteries, along with a special chapter on automotive technology trends related to software and artificial intelligence.

- Electric vehicle batteries - Global EV Outlook 2026 - IEA: Despite recent progress, sodium‑ion batteries remain constrained by lower energy density than lithium‑ion technologies, limiting their competitiveness with LFP batteries at current lithium prices. The latest sodium‑ion cells can reach up to 175 Wh/kg, compared with up to 205 Wh/kg for the latest generation of LFP batteries and 265 Wh/kg for lithium nickel cobalt manganese oxide (NMC) batteries. Their disadvantage is even greater in terms of volumetric energy density (Wh/l), which translates [...] At the same time, battery chemistry preference is shifting towards more affordable options, such as lithium iron phosphate (LFP) batteries, while interest in sodium-ion batteries is growing. This poses challenges for recycling businesses relying on the economic value of the recovered critical minerals, potentially requiring different business models and devoted regulations to ensure that end‑of‑life batteries are properly collected and processed. [...] The first sodium-ion battery-powered electric car was introduced in China only in late 2023, but the technology is now being scaled up by leading battery producers such as CATL and BYD. Sodium‑ion batteries perform significantly better at low temperatures than lithium‑ion batteries, particularly LFP chemistries. The latest generation of sodium-ion batteries can retain around 90% of nominal capacity at temperatures as low as -40°C, and can operate at temperatures as high as 70°C. For the largest
- 2026 Battery Technology Roadmap: LFP as Ballast, Solid-state Batteries in the Lead, Sodium-ion Batteries Securing a Spot: Sun Huajun, the CTO of BYD, publicly revealed that BYD will start the batch demonstration installation and application of all - solid - state batteries around 2027 and is expected to achieve large - scale installation around 2030. Data from the Gaogong Industry Research Institute shows that in the first four months of 2026, the expansion scale of the solid - state battery industry has exceeded 100GWh, with a planned investment of over 30 billion yuan and a total planned production capacity [...] - state battery is expected to start mass production in the second half of 2026. Guoxuan High - Tech presented its all - solid - state battery "Jinshi" with an energy density of 350Wh/kg and plans to start small - batch mass production by the end of 2026. [...] In response to the update of lithium iron phosphate technology, CATL, BYD, and Guoxuan High - Tech have fully mass - produced the fifth - generation lithium iron phosphate batteries. Institutions predict that the market share of this technology route will exceed 30% in 2026, becoming the mainstream technology platform for power batteries. At this CIBF2026, leading enterprises such as CATL, BYD, GAC, and EVE Energy have all presented new and mature technical solutions that can be directly
- Lithium-Ion Battery Materials Market Report 2026: Global Opportunities and Revenue Outlook to 2030: Market opportunities: Declining lithium-ion battery prices, growing research and development investment, battery performance upgrades, recycling, second-life material recovery, and circular supply models.
 Market challenges: Fluctuating raw material prices and supply-chain exposure.
 Product development and innovation: Emerging technologies and research and development activities across battery materials. [...] The global lithium-ion battery materials market is projected to grow from USD 48.29 billion in 2025 to USD 95.34 billion by 2030, registering a compound annual growth rate of 14.6% during the forecast period. Market growth is being driven by accelerating electrification, expanding electric vehicle production, renewable energy storage deployment, and sustained demand for consumer electronics. [...] 1. Crypto
2. AI

This is a paid press release. Contact the press release distributor directly with any inquiries.

GlobeNewswire

# Lithium-Ion Battery Materials Market Report 2026: Global Opportunities and Revenue Outlook to 2030

Research and Markets

5 min read

Company Logo

Company Logo

 

Electrolytes, lithium iron phosphate (LFP) chemistry and EV applications are expected to be the fastest-growing segments, while Europe will record the strongest regional growth
- Electric Battery Pack Guide 2026 Technology Types and Trends - LEAPENERGY: As a leading EV battery manufacturer, we’re investing R&D where it directly impacts U.S. customers and fleets:

 CTP/CTB and structural packs for lighter, safer, longer‑range vehicles
 800V‑ready high‑voltage battery packs that support ultra‑fast charging
 Advanced chemistries (high‑silicon, LFP+, sodium‑ion, semi‑solid) tuned for long cycle life and stable supply
 Smarter BMS and thermal management for better safety, higher usable capacity, and longer [...] Over the last decade, EV battery pack price per kWh has dropped from well over $1,000/kWh to well under $150/kWh at the pack level, and it’s still trending down. For U.S. buyers, that’s the main reason EV sticker prices are finally getting closer to gas cars. Long term, most forecasts put average pack prices under $80/kWh once large-scale LFP and next‑gen chemistries are fully ramped.

### What Actually Drives EV Battery Pack Cost

The real cost drivers are: [...] We’re running parallel validation programs with multiple cell partners so when these chemistries are commercially ready, our packs and BMS are ready too.

### Sodium-ion battery packs as an LFP alternative

Sodium‑ion battery packs are coming fast as a lower‑cost alternative to LFP:
- New Battery Technologies 2026 Are Changing The Industry. Here’s How: Packs for battery-electric cars came in at $99 per kWh, the second year under $100.

The catch for European drivers is geography. Average pack prices in China were $84 per kWh, while Europe ran about 56% higher. That’s a big reason an EV here still costs more than the petrol equivalent. BloombergNEF expects another small drop in 2026, and points to silicon, lithium-metal and solid-state as the next wave of cost cuts. [...] Line up the new battery technologies 2026, and the pattern is clear.

Solid-state has gone from a lab curiosity to a Mercedes crossing three countries on a single charge. Sodium-ion, the new electric car battery technology that looks like it’s built for a proper European winter, is about to put cheap cold-weather cars on sale. Silicon anodes are already in showrooms, quietly buying back range and minutes at the plug. [...] In February, CATL (the world’s largest battery maker) and Changan unveiled the Changan Nevo A06, the world’s first mass-production passenger EV running a sodium-ion pack, with mass production of the cells due by the end of 2026.

CATL’s Naxtra cell sits at about 175 Wh/kg today, close to lithium iron phosphate (LFP), and the company is targeting LFP-level density within three years to reach around 600 km of range.
- Global Advanced Rechargeable Battery Recycling Market Research 2026-2046, Profiles of 118 Leaders and Innovators Across Industrial Operators, Materials Giants and Emerging Circular Economy Startups: The global advanced rechargeable battery recycling market is evolving beyond lithium-ion dominance, with sodium-ion, solid-state, and other chemistries gaining ground. This transformative shift is driven by regulatory frameworks like the EU Battery Regulation and the US Inflation Reduction Act, enhancing domestic recycling capacities. The report from 2026 to 2046 offers in-depth insights into new market dynamics and forecasts, vital for stakeholders aiming to capture opportunities in this [...] Extensive quantitative forecasting covers global Li-ion recycling volumes (ktonnes) and revenues by cathode chemistry (NMC, LFP, NCA, LCO, LMFP), end-use application (EV, grid storage, consumer electronics), and region (China, Europe, North America, Rest of Asia-Pacific) from 2018 through 2046. The dominant structural trend - the inexorable shift of recycling feedstock toward LFP chemistry, projected to represent over 81% of Li-ion recycling input volumes by 2046 - and its profound implications [...] Battery technology landscape: Li-ion, sodium-ion, solid-state, vanadium redox flow, lithium-sulfur, lithium-metal, zinc-based, and aluminium-ion
 Li-ion cell chemistry, degradation mechanisms, cycle life, end-of-life pathways, and circular lifecycle
 EV battery longevity: real-world data from 22,700 vehicles and implications for recycling feedstock timelines
 Closed-loop EV battery value chain and the emerging replacement battery pack market

- Lithium Battery Prices in 2026: Current Costs, Trends & Key Changes - Lithium-ion Battery Manufacturer | BAKTH Battery [Official]: Based on the latest data from BloombergNEF and TrendForce as of June 2026, here are the core market figures:

 Global average lithium battery pack price: $110–$115/kWh
 China average battery pack price: $88–$92/kWh
 Stationary energy storage (BESS) pack price: $75–$82/kWh
 Battery-grade lithium carbonate: $178,300/ton, surging 120% compared with the $81,000/ton low in June 2025 [...] Global lithium battery annual sales will reach 5.5–8 TWh.
 Average battery pack price is forecast to drop to $60–$80/kWh (BloombergNEF).
 New technologies such as solid-state batteries will gradually increase energy density to 300–400 Wh/kg.

## 9. FAQs

 

##### Q1: Will lithium raw material prices continue to rise in 2026? [...] | 2026 (YTD) | $110–$115 | $88–$92 | $75–$82 | Lithium price rebound, strong storage demand |
| 2026 (EoY Forecast) | $112–$118 | $90–$95 | $78–$85 | Tight supply likely to trigger Q4 price hikes |
- Lithium-Ion Battery Materials Market Report 2026: Global Opportunities and Revenue Outlook to 2030: The global lithium-ion battery materials market is projected to grow from USD 48.29 billion in 2025 to USD 95.34 billion by 2030, registering a compound annual growth rate of 14.6% during the forecast period. Market growth is being driven by accelerating electrification, expanding electric vehicle production, renewable energy storage deployment, and sustained demand for consumer electronics. [...] Market opportunities: Declining lithium-ion battery prices, growing research and development investment, battery performance upgrades, recycling, second-life material recovery, and circular supply models.
 Market challenges: Fluctuating raw material prices and supply-chain exposure.
 Product development and innovation: Emerging technologies and research and development activities across battery materials. [...] Key Attributes

|  |  |
 --- |
| Report Attribute | Details |
| No. of Pages | 258 |
| Forecast Period | 2025-2030 |
| Estimated Market Value (USD) in 2025 | $48.29 Billion |
| Forecasted Market Value (USD) by 2030 | $95.34 Billion |
| Compound Annual Growth Rate | 14.6% |
| Regions Covered |  |

For more information about this report visit
- Battery Cost per kWh in 2026: What Drives It | BatteryMBA: Raw material costs have risen substantially since late 2025: battery-grade lithium carbonate roughly tripled from its mid-2025 trough of about 8,000 dollars per tonne. BloombergNEF still forecast a further decline to just under 105 dollars per kWh for 2026, but that forecast assumes the industry keeps absorbing input cost increases through contracts, hedging and continued LFP adoption.

## Related reading [...] Battery-grade lithium carbonate bottomed at around 8,000 US dollars per tonne in mid-2025 and then rebounded sharply. Chinese spot prices traded between roughly 150,000 and 181,500 yuan per tonne (about 21,000 to 25,500 dollars) through the first half of 2026, according to Shanghai Metals Market, with the peak in January. The International Energy Agency's Global EV Outlook 2026 noted that lithium prices at the start of 2026 stood at more than double their level a year earlier, while remaining [...] Chemistry accounts for a large part of the spread, with LFP packs averaging 81 dollars per kWh against 128 dollars per kWh for NMC.
 Regional differences are large: China averaged 84 dollars per kWh, while North America and Europe were 44 and 56 per cent higher respectively.
 Raw material prices rose sharply from late 2025 into 2026 without immediately passing through to pack prices, and that divergence is the defining uncertainty for cost forecasts made during 2026.
- EV Battery Statistics 2026: Prices, Chemistry, Market Share & Recycling Data - Axis Intelligence: The historical trajectory is as significant as the 2025 data point. Pack prices have declined 93% since 2010, when the BNEF survey recorded an average of approximately $1,436/kWh. For comparison:

| Year | Global Avg Pack Price ($/kWh) | YoY Change |
 --- 
| 2010 | $1,436 | — |
| 2020 | $137 | (benchmark) |
| 2023 | $139 | flat |
| 2024 | $115 | -20% |
| 2025 | $108 | -8% |
| 2026F | ~$105 | -3% (BNEF forecast) |

Source: BloombergNEF 2025 Lithium-Ion Battery Price Survey (December 9, 2025) [...] The global average lithium-ion battery pack price was $108/kWh in 2025, according to BloombergNEF’s 2025 Lithium-Ion Battery Price Survey published in December 2025. That is 8% below the 2024 average of $115/kWh and 93% below the 2010 starting point of $1,436/kWh. Battery electric vehicle packs specifically averaged $99/kWh — the second consecutive year below $100/kWh. Chinese manufacturers produced packs at $84/kWh on average, while North American and European regional averages were 44% and [...] 2. According to Axis Intelligence Research analysis of BloombergNEF’s 2025 Lithium-Ion Battery Price Survey, the average lithium-ion pack price fell to $108/kWh in 2025 — 93% below the $1,436/kWh recorded in 2010, and the BEV-specific pack price held below $100/kWh for the second consecutive year at $99/kWh.
- EV battery costs set to spike as raw material shortages drags on: The expected surge in battery costs could drive the price of EVs sold in 2026 up anywhere between $1,500 and $3,000 per vehicle, E Source predicts. The firm has also reduced its EV sales projections for 2026 by 5% to 10%.

By then, EV sales are projected to top 2 million annually in the U.S., according to the latest forecast from consulting firm LMC Automotive. Automakers are expected to introduce dozens of electric models as more Americans embrace the idea of going electric.

- Electric vehicle batteries - Global EV Outlook 2026 - IEA: all-solid-state battery-powered vehicle by 2028 – and the Chinese firm BYD, which plans to sell its first EV using all-SSB from 2027 and to begin mass production from 2030. Korea’s Samsung has set similar timelines for all-SSB production. Developments are also continuing in the United States. QuantumScape tested its SSB in a motorcycle in 2025, while Factorial Energy announced plans to list on public markets after reporting a real‑world test of well over 1 000 kilometres of range. [...] The first sodium-ion battery-powered electric car was introduced in China only in late 2023, but the technology is now being scaled up by leading battery producers such as CATL and BYD. Sodium‑ion batteries perform significantly better at low temperatures than lithium‑ion batteries, particularly LFP chemistries. The latest generation of sodium-ion batteries can retain around 90% of nominal capacity at temperatures as low as -40°C, and can operate at temperatures as high as 70°C. For the largest [...] In 2025, LFP batteries accounted for over 55% of EV batteries deployed globally, up from nearly 50% in 2024. Deployment remains heavily concentrated in China, but uptake is also expanding rapidly in other EMDEs. LFP batteries now power two-thirds of all electric car sales in these economies – double the share in 2023 – driven by imports of Chinese-manufactured vehicles and batteries.

### Electric vehicle battery sales share by chemistry and region, 2023-2025

  Open

####

- Lithium-Ion Battery Pack Prices Fall to $108 Per Kilowatt-Hour, Despite Rising Metal Prices: BloombergNEF | BloombergNEF: December 9, 2025

New York, December 9, 2025 – lithium-ion battery pack prices have dropped 8% since 2024 to a record low of $108 per kilowatt-hour, according to latest analysis by research provider BloombergNEF (BNEF). Continued cell manufacturing overcapacity, intense competition and the ongoing shift to lower-cost lithium iron phosphate (LFP) batteries helped drive down pack prices despite an increase in battery metal costs according to BNEF’s 2025 Lithium-Ion Battery Price Survey. [...] BNEF expects pack prices to decrease again in 2026, based on its near-term outlook, as raw material prices face upward pressure but adoption of low-cost LFP continues to spread. Over the longer term, ongoing investment in R&D, manufacturing efficiency and supply chain expansion is expected to support further technology improvements and cost reductions. Emerging technologies, including silicon and lithium metal anodes, solid-state electrolytes, new cathode materials and new cell manufacturing [...] BNEF’s industry-leading battery price survey covers multiple battery end-uses, including different types of electric vehicles and stationary storage projects. Each sector typically requires different cells and packs to meet distinct performance metrics, leading to varied pricing dynamics across these use cases. Battery pack prices for stationary storage dropped to $70/kWh in 2025, 45% lower than in 2024. This is the sharpest drop across all segments, making stationary storage the lowest-priced
- Clean Energy: Lithium-Ion Battery Pack Prices Hit Record Low of $139/kWh. November 26, 2023. BloombergNEF's annual battery price survey finds a 14% drop from 2022 to 2023Read more
- Insights | Page 49 of 71 | BloombergNEF: ## Flagship reports

All Reports

REPORT

#### Climatescope

REPORT

#### NetZero Pathfinders

REPORT

#### Electric Vehicle Outlook

REPORT

#### Transition Metals Outlook

REPORT

#### Energy Transition Investment Trends

REPORT

#### New Energy Outlook

#### Climatescope

#### NetZero Pathfinders

#### Electric Vehicle Outlook

#### Transition Metals Outlook

#### Energy Transition Investment Trends

#### New Energy Outlook

## Stay up-to-date [...] # Insights

# Insights

Access the latest perspectives on the energy transition from BNEF experts through our comprehensive range of research reports and analysis, each substantiated by transparent data and methodologies.

#### Our insights, your inbox

Get the latest BNEF research, news, webinars and podcasts in our monthly email newsletter.

Sign Up

## Explore all topics

Clean Energy

Clean Transport

Commodities

Data Centers

Finance

Industry and Buildings

Nature and Agriculture [...] ## Stay up-to-date

Keep up with the trends shaping the energy transition with our podcast and events.

### Switched On podcast

The future of energy, transport, sustainability and more, uncovered weekly.

View all episodes

### BNEF Events

Take part in the global conversation on the shift to a lower-carbon economy.

Find events

### Discover BNEF's Climate Week and COP Public Research, freely available to non-subscribers.

## All insights

- Electric vehicle batteries - Global EV Outlook 2026 - IEA: ## Battery recycling

### Recycling will remain dominated by production scrap until end‑of‑life batteries take the lead in the mid‑2030s [...] all-solid-state battery-powered vehicle by 2028 – and the Chinese firm BYD, which plans to sell its first EV using all-SSB from 2027 and to begin mass production from 2030. Korea’s Samsung has set similar timelines for all-SSB production. Developments are also continuing in the United States. QuantumScape tested its SSB in a motorcycle in 2025, while Factorial Energy announced plans to list on public markets after reporting a real‑world test of well over 1 000 kilometres of range. [...] Battery recycling is crucial for the long‑term sustainability of the battery industry and is likely to become an important future source of critical minerals, strengthening battery supply chain security and resilience. Today, recycling already plays an important role in supporting the battery value chain, primarily through the recovery of production scrap generated during the manufacturing of battery cells and components. However, beyond this, the contribution of recycling to meeting battery
- Executive summary - Recycling of Critical Minerals - IEA: Battery recycling capacity is outpacing available feedstock, although the picture varies by region. If all announced projects come online as scheduled, global recycling capacity in 2030 could be seven times the available feedstock, even with the rapid levels of battery deployment achieved in the APS. However, the picture changes rapidly after 2030 as EVs reach end of life and feedstock availability increases sharply, surpassing announced capacity by 60% by 2040. There are major regional [...] Manufacturing scrap currently dominates the feedstock for battery recycling, but this balance shifts rapidly towards end-of-life EV batteries. Scrap from manufacturing processes still accounts for two-thirds of available recycling feedstock in 2030. From 2035 onwards, however, end-of-life EV and storage batteries take over as the largest source and represent over 90% of available feedstock by 2050. Lithium-ion batteries are one of the key technologies pushing up demand for critical minerals, [...] Relatively slow development of midstream battery supply projects is a major uncertainty for recyclers. While battery cell production projects are expanding in regions such as Europe and the United States (US), plans for the midstream supply chain such as precursor cathode active material (pCAM) and cathode active material (CAM) remain limited. By 2030, nearly 90% of these capacities are expected to remain concentrated in China, reducing the security benefits of recycling as recyclers need to
- How can innovation help secure future battery markets and mineral supplies? – Analysis - IEA: While solid-state battery cells are already being produced, their integration into EV battery packs remains challenging. Among the most advanced, Chinese firm BYD plans to sell its first EV using all-solid-state batteries from 2027, with mass production following in 2030. Japan’s Toyota and Korea’s Samsung have set similar timelines.

- Lithium-Ion Battery Pack Prices Fall to $108 Per Kilowatt-Hour, Despite Rising Metal Prices: BloombergNEF | BloombergNEF: December 9, 2025

New York, December 9, 2025 – lithium-ion battery pack prices have dropped 8% since 2024 to a record low of $108 per kilowatt-hour, according to latest analysis by research provider BloombergNEF (BNEF). Continued cell manufacturing overcapacity, intense competition and the ongoing shift to lower-cost lithium iron phosphate (LFP) batteries helped drive down pack prices despite an increase in battery metal costs according to BNEF’s 2025 Lithium-Ion Battery Price Survey. [...] BNEF’s industry-leading battery price survey covers multiple battery end-uses, including different types of electric vehicles and stationary storage projects. Each sector typically requires different cells and packs to meet distinct performance metrics, leading to varied pricing dynamics across these use cases. Battery pack prices for stationary storage dropped to $70/kWh in 2025, 45% lower than in 2024. This is the sharpest drop across all segments, making stationary storage the lowest-priced [...] PRESS RELEASE

# Lithium-Ion Battery Pack Prices Fall to $108 Per Kilowatt-Hour, Despite Rising Metal Prices: BloombergNEF

 Clean Energy
 Clean Transport

December 9, 2025
- BNEF: Lithium-ion battery pack prices fall to $108/kWh, stationary storage becomes lowest price segment - Energy Storage: According to BNEF, battery pack prices for stationary storage fell to $70/kWh in 2025, a 45% decrease from 2024. This represents the steepest decline among all lithium-ion battery use cases and and makes stationary storage the cheapest category for the first time.

, prices have fallen 8% since 2024 to $108/kWh, making them 93% lower than in 2010. [...] Despite an increase in battery metal costs, continued cell manufacturing overcapacity, intense competition and the ongoing shift to lower-cost lithium iron phosphate (LFP) batteries helped drive down pack prices, according to BNEF’s 2025 Lithium-Ion Battery Price Survey. [...] BNEF’s battery price survey – which spans multiple end uses, including various electric-vehicle types and stationary-storage applications – reveals pronounced differences across sectors. Battery pack prices for stationary storage fell to $70/kWh in 2025, 45% lower than in 2024. This was the steepest decline of any segment, making stationary storage the lowest-priced category for the first time. In the transport sector, battery-electric vehicle packs were the cheapest at $99/kWh, marking the

### 🤖 Researcher Agent's Reaserach
- **Cheaper batteries, with price risk:** Battery-electric vehicle packs averaged $99/kWh in 2025. Manufacturing improvements and competition support further reductions, but higher lithium and other mineral prices could slow or reverse them. [IEA](https://www.iea.org/reports/global-ev-outlook-2026/electric-vehicle-batteries)
- **LFP is becoming the mainstream choice:** Lithium iron phosphate batteries accounted for over 55% of EV batteries deployed globally in 2025. They offer lower costs, while nickel-rich batteries remain attractive where higher energy density and longer range matter most. [IEA](https://www.iea.org/reports/global-ev-outlook-2026/electric-vehicle-batteries)
- **Sodium-ion is emerging for lower-cost EVs:** It avoids lithium and can perform well in cold weather, but generally has lower energy density than lithium-ion. Scaling is underway; competitiveness will depend partly on lithium prices. [IEA](https://www.iea.org/reports/global-ev-outlook-2026/electric-vehicle-batteries)
- **Solid-state remains a longer-term prospect:** It promises improvements in energy density, but manufacturing and integration remain challenges. Companies are targeting initial vehicles in the late 2020s and broader production around 2030; those timelines are plans, not guarantees. [IEA](https://www.iea.org/reports/global-ev-outlook-2026/electric-vehicle-batteries)
- **Recycling will grow, but economics are changing:** Manufacturing scrap dominates recycling today; end-of-life batteries are expected to become the main source around the mid-2030s. The shift toward lower-value LFP materials may require new recycling business models. [IEA](https://www.iea.org/reports/global-ev-outlook-2026/electric-vehicle-batteries)

In [26]:
analyst_result = await Runner.run(analyst_agent, researcher_result.final_output.summary)
print_markdown(f"### 🤖 Analyst Agent's Answer\n{analyst_result.final_output.summary}")

### 🤖 Analyst Agent's Answer
EV batteries are trending toward lower costs and a wider mix of chemistries: LFP accounted for over 55% of global EV battery deployment in 2025, while nickel-rich batteries retain an advantage where range matters. Sodium-ion could strengthen the low-cost segment, but its competitiveness depends partly on lithium prices; solid-state batteries remain a longer-term prospect with uncertain production timelines.

The main risks are commodity-price volatility and recycling economics. Higher mineral prices could slow or reverse battery cost declines, while LFP’s lower-value recovered materials may require new recycling business models as end-of-life batteries become the main feedstock around the mid-2030s.